# raw → `df.parquet`

`data-v2/raw/`만 읽어서 모델 입력 `data-v2/input/df.parquet`를 만듭니다. 로직은 모두 이 노트북 안에 있고, 다른 모듈이나 중간 캐시(processed/)는 쓰지 않습니다. 실행할 때마다 SUVI NetCDF에서 다시 추출합니다(12 worker로 약 30분).

| 단계 | 내용 |
|---|---|
| 0 | 설정과 시간 칸 규칙 |
| 1 | ACE 월별 txt → 시간 칸 표, 정답 `y` |
| 2 | ACE 피처 |
| 3 | SUVI NetCDF → 프레임별 셀 통계·면적 |
| 4 | SUVI 피처(프레임, `img_age_h`, strip) |
| 5 | ENLIL 피처 |
| 6 | df 조립(446열) |
| 6b | 추가 피처 20열(DONKI 알림, NOAA SRS·예보 토의, ACE 일별 1분 파일, WSA-Enlil 궤적, 보조 분류기) → `df.parquet` 저장 |
| 7 | 검증 |

**규칙**
- 입수지연·결측·이상치 처리는 하지 않습니다. 앞값 채움, 이전 프레임으로 되돌아가기, stale/cover 마스크, 값 자르기(clip)·로그 변환이 없고, 피처는 원문 값에서 계산만 합니다.
- **시간 칸**: 시각 t는 칸 `floor(t, 1h) + 1h`에 들어갑니다. 즉 [H−1h, H) → H이고, 19:49·19:59·19:00은 모두 20시 칸입니다. 한 칸에 여러 개면 가장 늦은 것을 씁니다. 발표 시각이 따로 없는 소스도 데이터의 시각에 같은 규칙을 씁니다. origin O의 피처는 칸 ≤ O만 씁니다.
- **정답 `y`**: 공식 채점(`metric/metric.ipynb`)과 같습니다. target_time T의 `y`는 T 라벨 SWEPAM 속도이고, 시각을 이동하지 않습니다. `status == 0`이고 유한한 양수일 때만 값이 있고, 나머지는 NaN이며 채우지 않습니다.
- **ACE 피처 값**: `status ≠ 0`이거나 결측 표시값(제품 헤더의 `Missing data values`)이면 NaN입니다.
- **6b절 예외**: 추가 20열은 원 정의를 그대로 따릅니다. 공개 시각(발행·`Created`·`Last-Modified`)이 origin 이전인 원문만 쓰고, 결측은 `-999`로 표시하며, `ace_proton_ram_pressure_npa`는 h별 앞값 채움 뒤의 n, v로 계산합니다. `onset700_hazard`는 2025-09-23 이전 학습 y로 적합한 보조 분류기의 출력입니다.
- 이름에 `ch`를 쓰지 않습니다. SUVI에서 나온 열은 `suvi_`로 시작합니다(`strip_*`, `img_age_h`는 예전 이름 그대로).

## 0. 설정

In [1]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

import json, re, time, warnings, multiprocessing
from concurrent.futures import ProcessPoolExecutor
from datetime import datetime, timezone
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from scipy import ndimage

warnings.filterwarnings("ignore", category=RuntimeWarning)   # 전부 NaN인 창의 평균 등
pd.set_option("display.width", 200, "display.max_columns", 40)

ROOT = Path("/home/t-lab01/sunrun")
DATA = ROOT / "data-v2"
ACE_DIR = DATA / "raw/ACE/monthly-txt"
NC_DIR = DATA / "raw/SUVI/nc/g18"
ENLIL_DIR = DATA / "raw/ENLIL/donki"
OUT = Path(os.environ.get("DF_OUT", DATA / "input"))       # df.parquet, MANIFEST.json
SMOKE = os.environ.get("DF_SMOKE") == "1"                   # 짧은 구간만 (확인용)
WORKERS = int(os.environ.get("DF_WORKERS", "12"))

H = 72
MASK_HOURS = 144
HOUR = pd.Timedelta(hours=1)
PERIODS = {  # 이름 -> (첫 query origin, 마지막 query origin, 학습 cutoff)
    "F1": ("2025-12-31T23:00Z", "2026-03-28T23:00Z", "2025-12-29T00:00Z"),
    "F2": ("2026-03-31T23:00Z", "2026-06-27T23:00Z", "2026-03-29T00:00Z"),
    "F3": ("2026-06-30T23:00Z", "2026-09-26T23:00Z", "2026-06-29T00:00Z"),
}
QUERY_SPAN = ("2025-12-31T23:00Z", "2026-09-26T23:00Z")
TRAIN_SPAN = ("2022-10-01T00:00Z", str(pd.Timestamp(PERIODS["F3"][2]) - pd.Timedelta(hours=MASK_HOURS)))
if SMOKE:
    TRAIN_SPAN = ("2026-05-20T00:00Z", "2026-05-21T23:00Z")
    QUERY_SPAN = ("2026-05-28T00:00Z", "2026-05-29T23:00Z")

train_origins = pd.date_range(*TRAIN_SPAN, freq="h")
query_origins = pd.date_range(*QUERY_SPAN, freq="h")
all_origins = train_origins.union(query_origins)


def bin_hour(t):
    # 시각 -> 시간 칸: [H-1h, H) -> H
    return pd.DatetimeIndex(pd.to_datetime(t, utc=True)).floor("h") + HOUR


for raw_t, want in [("2026-05-01T19:49Z", "2026-05-01T20:00Z"), ("2026-05-01T19:59:59Z", "2026-05-01T20:00Z"),
                    ("2026-05-01T19:00Z", "2026-05-01T20:00Z"), ("2026-05-01T20:00Z", "2026-05-01T21:00Z")]:
    assert bin_hour([raw_t])[0] == pd.Timestamp(want), (raw_t, want)

print("SMOKE" if SMOKE else "FULL", "| train origin", len(train_origins), TRAIN_SPAN, "| query origin", len(query_origins), QUERY_SPAN)
print("출력:", OUT, "| workers:", WORKERS)

FULL | train origin 32665 ('2022-10-01T00:00Z', '2026-06-23 00:00:00+00:00') | query origin 6457 ('2025-12-31T23:00Z', '2026-09-26T23:00Z')
출력: /home/t-lab01/sunrun/data-v2/input | workers: 12


## 1. ACE: 월별 txt → 시간 칸 표, 정답 `y`

NASA 월별 1h 파일 5종(`swepam, mag, epam, sis, loc`)을 읽습니다. 각 행의 라벨 시각은 `YR MO DA HHMM`(UT)입니다.
- 피처용 표: 라벨 시각 t → 칸 `bin_hour(t)` = t + 1h. 값은 그 값의 status가 0이 아니거나 결측 표시값이면 NaN입니다(loc에는 status 열이 없음).
- `y`용 표: 라벨 시각 그대로, SWEPAM `status == 0`이고 유한한 양수인 속도.
- 같은 라벨 시각이 두 번 나오면(월 경계 등) 나중 파일의 행을 쓰고, 값이 서로 다른 중복은 개수를 기록합니다.

In [2]:
# 제품 -> (시각 뒤 필드 순서, 결측 표시값). 필드: (열 이름 | None, status 열 이름 | None)
# 시각 필드 6개: YR MO DA HHMM MJD SEC
ACE_SPEC = {
    "swepam": ([("S", None), ("ace_density_cm3", "S"), ("ace_speed_kms", "S"), ("ace_temperature_k", "S")],
               {"ace_density_cm3": -9999.9, "ace_speed_kms": -9999.9, "ace_temperature_k": -1.0e5}),
    "mag": ([("S", None), ("ace_bx_gsm_nt", "S"), ("ace_by_gsm_nt", "S"), ("ace_bz_gsm_nt", "S"), ("ace_bt_nt", "S"),
             ("ace_lat_gsm_deg", "S"), ("ace_lon_gsm_deg", "S")], -999.9),
    "epam": ([("Se", None), ("ace_epam_e_channel1", "Se"), ("ace_epam_e_channel2", "Se"), ("Sp", None),
              ("ace_epam_p_channel1", "Sp"), ("ace_epam_p_channel2", "Sp"), ("ace_epam_p_channel3", "Sp"),
              ("ace_epam_p_channel4", "Sp"), ("ace_epam_p_channel5", "Sp"), ("ace_epam_anis_index", "Sp")],
             {"ace_epam_anis_index": -1.0, "*": -1.0e5}),
    "sis": ([("S10", None), ("ace_sis_gt10_flux", "S10"), ("S30", None), ("ace_sis_gt30_flux", "S30")], -1.0e5),
    "loc": ([("ace_x_gse_re", None), ("ace_y_gse_re", None), ("ace_z_gse_re", None)], -999.9),
}
ACE_LABEL = {  # 원문 헤더의 이름 (문서용)
    "ace_density_cm3": "SWEPAM proton density", "ace_speed_kms": "SWEPAM bulk speed", "ace_temperature_k": "SWEPAM ion temperature",
    "ace_bx_gsm_nt": "MAG Bx GSM", "ace_by_gsm_nt": "MAG By GSM", "ace_bz_gsm_nt": "MAG Bz GSM", "ace_bt_nt": "MAG Bt",
    "ace_lat_gsm_deg": "MAG lat GSM", "ace_lon_gsm_deg": "MAG long GSM",
    "ace_epam_e_channel1": "EPAM electron 38-53 keV", "ace_epam_e_channel2": "EPAM electron 175-315 keV",
    "ace_epam_p_channel1": "EPAM proton 47-68 keV", "ace_epam_p_channel2": "EPAM proton 115-195 keV",
    "ace_epam_p_channel3": "EPAM proton 310-580 keV", "ace_epam_p_channel4": "EPAM proton 795-1193 keV",
    "ace_epam_p_channel5": "EPAM proton 1060-1900 keV", "ace_epam_anis_index": "EPAM anisotropy index",
    "ace_sis_gt10_flux": "SIS proton > 10 MeV", "ace_sis_gt30_flux": "SIS proton > 30 MeV",
    "ace_x_gse_re": "LOC X GSE", "ace_y_gse_re": "LOC Y GSE", "ace_z_gse_re": "LOC Z GSE",
}


def read_ace(product):
    fields, missing = ACE_SPEC[product]
    names = [n for n, _ in fields]
    rows, files = [], sorted(ACE_DIR.glob(f"*_ace_{product}_1h.txt"))
    for order, path in enumerate(files):
        for line in path.read_text().splitlines():
            if not line or line[0] in ":#":
                continue
            parts = line.split()
            if len(parts) != 6 + len(fields):
                raise ValueError(f"{path.name}: 필드 수 {len(parts)}: {line!r}")
            rows.append([order] + parts[:4] + parts[6:])
    raw = pd.DataFrame(rows, columns=["file", "yr", "mo", "da", "hhmm"] + names)
    hhmm = raw["hhmm"].astype(int)
    t = pd.to_datetime(dict(year=raw.yr.astype(int), month=raw.mo.astype(int), day=raw.da.astype(int)), utc=True)
    raw.insert(0, "label_utc", t + pd.to_timedelta(hhmm // 100, unit="h") + pd.to_timedelta(hhmm % 100, unit="m"))
    vals = raw[names].astype(float)
    vals.insert(0, "label_utc", raw["label_utc"])
    vals.insert(1, "file", raw["file"])
    conflict = int(vals.drop(columns="file").drop_duplicates().duplicated("label_utc").sum())
    vals = vals.sort_values(["label_utc", "file"]).drop_duplicates("label_utc", keep="last").set_index("label_utc")
    clean = pd.DataFrame(index=vals.index)
    for name, status in fields:
        if not name.startswith("ace_"):
            continue
        v = vals[name].copy()
        miss = missing.get(name, missing.get("*")) if isinstance(missing, dict) else missing
        bad = np.isclose(v, miss, rtol=0, atol=1e-6) | ~np.isfinite(v)
        if status is not None:
            bad |= vals[status].ne(0)
        clean[name] = v.where(~bad)
    return clean, vals, {"files": len(files), "rows": len(rows), "labels": len(vals), "conflicting_duplicates": conflict}


ace_parts, ace_raw, ace_info = {}, {}, {}
for product in ACE_SPEC:
    ace_parts[product], ace_raw[product], ace_info[product] = read_ace(product)
ace_label = pd.concat(ace_parts.values(), axis=1).sort_index()      # 라벨 시각 기준, status/결측 표시 -> NaN
assert ace_label.index.is_unique

# 피처용: 라벨 시각 -> 칸 (라벨이 모두 정각이라 칸 = 라벨 + 1h, 한 칸에 한 행)
ace_bin = ace_label.copy()
ace_bin.index = bin_hour(ace_bin.index)
ace_bin = ace_bin[~ace_bin.index.duplicated(keep="last")]
ace_bin = ace_bin.reindex(pd.date_range(ace_bin.index.min(), ace_bin.index.max(), freq="h"))

# y용: 라벨 시각 그대로, SWEPAM status == 0 이고 유한한 양수
sw = ace_raw["swepam"]
y_series = sw["ace_speed_kms"].where(sw["S"].eq(0) & np.isfinite(sw["ace_speed_kms"]) & (sw["ace_speed_kms"] > 0))
y_series = y_series.reindex(pd.date_range(y_series.index.min(), y_series.index.max(), freq="h"))

display(pd.DataFrame(ace_info).T)
print("ACE 라벨 시각:", ace_label.index.min(), "~", ace_label.index.max(), "| 칸:", ace_bin.index.min(), "~", ace_bin.index.max())
print("y 유효:", int(y_series.notna().sum()), "/", len(y_series), f"({y_series.notna().mean():.1%})")
display(pd.DataFrame({"원문": ACE_LABEL, "NaN 비율": ace_label.isna().mean().round(3)}))

,files,rows,labels,conflicting_duplicates
swepam,52,37305,37305,0
mag,52,37305,37305,0
epam,52,37305,37305,0
sis,52,37305,37305,0
loc,52,37320,37320,0


ACE 라벨 시각: 2022-08-01 00:00:00+00:00 ~ 2026-11-02 23:00:00+00:00 | 칸: 2022-08-01 01:00:00+00:00 ~ 2026-11-03 00:00:00+00:00
y 유효: 25169 / 37305 (67.5%)


,원문,NaN 비율
ace_density_cm3,SWEPAM proton density,0.323
ace_speed_kms,SWEPAM bulk speed,0.326
ace_temperature_k,SWEPAM ion temperature,0.326
ace_bx_gsm_nt,MAG Bx GSM,0.174
ace_by_gsm_nt,MAG By GSM,0.174
ace_bz_gsm_nt,MAG Bz GSM,0.174
ace_bt_nt,MAG Bt,0.174
ace_lat_gsm_deg,MAG lat GSM,0.174
ace_lon_gsm_deg,MAG long GSM,0.174
ace_epam_e_channel1,EPAM electron 38-53 keV,0.144


## 2. ACE 피처

origin O에서 칸 ≤ O만 씁니다(칸 O = 라벨 O−1h 행).
- 22개 변수마다 `<변수>_last`(칸 O), `_m6`(칸 O−5…O 평균), `_m24`(칸 O−23…O 평균). 평균은 창 안의 NaN을 빼고 계산하고, 창이 전부 NaN이면 NaN입니다.
- 속도 파생: `v_lag{0,1,3,6,12,24,48,72,120}` = 칸 O−k의 속도, `v_m{6,24,72}`(평균), `v_sd{6,24,72}`(표준편차, 값 2개 이상).
- `speed_d24 = ace_speed_kms_m6(O) − ace_speed_kms_m6(O−24h)`, `epam_p1_d6 = ace_epam_p_channel1_m6(O) − ace_epam_p_channel1_m6(O−6h)`.
- 예전 정의와 다른 점: `_last`의 앞값 채움(최대 6h)을 하지 않고, EPAM/SIS의 log10과 1e-3 clip을 하지 않습니다. 속도는 채워진 정답 시계열이 아니라 위 ACE 칸 값(status 0)입니다.

In [3]:
ACE_VARS = list(ace_label.columns)
f = {}
for c in ACE_VARS:
    x = ace_bin[c]
    f[f"{c}_last"] = x
    f[f"{c}_m6"] = x.rolling(6, min_periods=1).mean()
    f[f"{c}_m24"] = x.rolling(24, min_periods=1).mean()
speed = ace_bin["ace_speed_kms"]
v = {f"v_lag{k}": speed.shift(k) for k in (0, 1, 3, 6, 12, 24, 48, 72, 120)}
for w in (6, 24, 72):
    v[f"v_m{w}"] = speed.rolling(w, min_periods=1).mean()
    v[f"v_sd{w}"] = speed.rolling(w, min_periods=2).std()
f["speed_d24"] = f["ace_speed_kms_m6"] - f["ace_speed_kms_m6"].shift(24)
f["epam_p1_d6"] = f["ace_epam_p_channel1_m6"] - f["ace_epam_p_channel1_m6"].shift(6)
ace_feat = pd.concat([pd.DataFrame(v), pd.DataFrame(f)], axis=1).reindex(all_origins)
print("ACE 피처:", ace_feat.shape[1], "열 | origin", len(ace_feat))
ace_feat.iloc[:3, :12]

ACE 피처: 83 열 | origin 34968


,v_lag0,v_lag1,v_lag3,v_lag6,v_lag12,v_lag24,v_lag48,v_lag72,v_lag120,v_m6,v_sd6,v_m24
2022-10-01 00:00:00+00:00,NaN,NaN,414.0,451.8,501.7,484.6,462.4,605.0,300.9,426.000000,16.970563,490.83125
2022-10-01 01:00:00+00:00,445.5,NaN,NaN,NaN,NaN,478.9,466.0,611.6,NaN,432.500000,16.454483,488.74375
2022-10-01 02:00:00+00:00,430.9,445.5,NaN,438.0,459.1,476.9,461.2,NaN,293.9,430.133333,15.763988,485.86875


## 3. SUVI: NetCDF → 프레임별 셀 통계·면적

`raw/SUVI/nc/g18/`의 GOES-18 Fe195 L1b 파일을 씁니다. `fetch.py`가 시간마다 한 장씩 저장했고, 파일 이름의 `_e…`(관측 끝 `obs_end`)로 칸 `bin_hour(obs_end)`를 정합니다. 한 칸에 파일이 여러 개면 `obs_end`가 가장 늦은 것을 씁니다.

프레임마다 두 가지를 한 번에 계산합니다(유효 픽셀 = 디스크 0.98 R☉ 안, `DQF == 0`, 유한한 비음수 radiance).
- **셀 통계 148열** `suvi_disk_*`, `suvi_lonII_latJ_*`: 디스크 전체와 경도 12칸 × 위도 3칸마다 log1p radiance의 중앙값(`log_median`)·10% 분위(`log_q10`), 어두운 픽셀 비율(`dark_fraction`, 디스크 중앙값의 0.5배 미만), 유효 픽셀 비율(`valid_fraction`).
- **면적 180열** `suvi_t035_*`, `suvi_t045_*`, `suvi_t055_*`: 디스크 중앙값의 0.35/0.45/0.55배보다 어두운 영역(opening·closing 3×3, 면적 0.0005 이상)의 면적, 개수, 상위 3개 영역의 면적·위치·폭, 중앙 자오선 ±15°/±30°와 적도 ±30° 안의 면적, 경도×위도 36칸 면적.

계산식은 `leaderboard/model-release-publish/experiments/xgboost_suvi_fusion/{features,coronal_holes}.py`에서 옮겼고 열 이름만 바꿨습니다. 파일을 읽지 못하면 그 칸은 NaN이 되고, 파일 목록은 MANIFEST에 남깁니다.

In [4]:
SUVI_NAME = re.compile(r"_s(\d{4})(\d{3})(\d{2})(\d{2})(\d{2})(\d)_e(\d{4})(\d{3})(\d{2})(\d{2})(\d{2})(\d)_c")


def stamp(g):
    year, doy, hh, mm, ss, tenth = (int(x) for x in g)
    return pd.Timestamp(year=year, month=1, day=1, tz="UTC") + pd.Timedelta(days=doy - 1, hours=hh, minutes=mm,
                                                                             seconds=ss, milliseconds=100 * tenth)


def read_suvi_frame(path):
    # RAD를 물리 radiance로 한 번만 디코딩 (h5py, 자동 scale 없음). fill 값은 NaN
    with h5py.File(path, "r") as nc:
        var = nc["RAD"]
        raw = np.asarray(var[...])
        rad = raw.astype(np.float64)
        invalid = ~np.isfinite(rad)
        for name in ("_FillValue", "missing_value"):
            if name in var.attrs:
                invalid |= raw == np.asarray(var.attrs[name]).reshape(-1)[0]
        rad = rad * float(np.asarray(var.attrs.get("scale_factor", 1)).item())
        rad += float(np.asarray(var.attrs.get("add_offset", 0)).item())
        rad[invalid] = np.nan
        header = dict(nc.attrs)
        for key in ("CRPIX1", "CRPIX2", "RSUN", "CROTA", "SOLAR_B0", "CDELT1", "CDELT2", "PC1_1", "PC1_2", "PC2_1", "PC2_2"):
            if key in nc:
                header[key] = np.asarray(nc[key][...]).item()
        return rad.squeeze(), np.asarray(nc["DQF"][...]).squeeze(), header


def cell_stats(rad, dqf, header):
    # 디스크와 경도 12 x 위도 3 셀의 log1p radiance 통계 (148열)
    rad, dqf = np.asarray(rad, dtype=float), np.asarray(dqf)
    h = {str(k).upper(): v for k, v in header.items()}

    def number(key, default=None):
        value = h.get(key, default)
        if value is None:
            raise ValueError(f"missing SUVI geometry: {key}")
        return float(np.asarray(value).item())

    yy, xx = np.indices(rad.shape, dtype=float)
    xx -= number("CRPIX1") - 1
    yy -= number("CRPIX2") - 1
    theta = np.deg2rad(number("CROTA", 0))
    matrix = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
    if all(k in h for k in ("PC1_1", "PC1_2", "PC2_1", "PC2_2")):
        matrix = np.array([[number("PC1_1"), number("PC1_2")], [number("PC2_1"), number("PC2_2")]])
    rsun = number("RSUN")
    if not np.isfinite(rsun) or rsun <= 0:
        raise ValueError("RSUN must be positive")
    x = (matrix[0, 0] * xx + matrix[0, 1] * yy) / rsun
    y = (matrix[1, 0] * xx + matrix[1, 1] * yy) / rsun
    disk = x * x + y * y <= 0.98 ** 2
    valid = disk & (dqf == 0) & np.isfinite(rad) & (rad >= 0)
    z = np.sqrt(np.maximum(0, 1 - x * x - y * y))
    b0 = np.deg2rad(number("SOLAR_B0", 0))
    lat = np.arcsin(np.clip(y * np.cos(b0) + z * np.sin(b0), -1, 1))
    lon = np.arctan2(x, z * np.cos(b0) - y * np.sin(b0))
    longitude = np.clip(np.floor((lon + np.pi / 2) / np.pi * 12), 0, 11).astype(int)
    latitude = np.clip(np.floor((lat + np.pi / 2) / np.pi * 3), 0, 2).astype(int)
    reference = np.median(rad[valid]) if valid.any() else np.nan
    result = {}
    for name, region in [("disk", disk)] + [(f"lon{i:02d}_lat{j}", disk & (longitude == i) & (latitude == j))
                                            for i in range(12) for j in range(3)]:
        values = rad[valid & region]
        log = np.log1p(values)
        result[f"suvi_{name}_log_median"] = float(np.median(log)) if len(log) else np.nan
        result[f"suvi_{name}_log_q10"] = float(np.quantile(log, .1)) if len(log) else np.nan
        result[f"suvi_{name}_dark_fraction"] = float(np.mean(values < .5 * reference)) if len(log) else np.nan
        result[f"suvi_{name}_valid_fraction"] = float(len(values) / region.sum()) if region.any() else 0.
    return result


THRESHOLDS = (0.35, 0.45, 0.55)


def dark_geometry(shape, h):
    rsun = h["RSUN"]
    if not np.isfinite(rsun) or rsun <= 0:
        raise ValueError("RSUN must be positive")
    y, x = np.indices(shape, dtype=float)
    x -= h["CRPIX1"] - 1
    y -= h["CRPIX2"] - 1
    theta = np.deg2rad(h.get("CROTA", 0))
    matrix = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
    if all(k in h for k in ("PC1_1", "PC1_2", "PC2_1", "PC2_2")):
        matrix = np.array([[h["PC1_1"], h["PC1_2"]], [h["PC2_1"], h["PC2_2"]]])
    xx = (matrix[0, 0] * x + matrix[0, 1] * y) / rsun
    yy = (matrix[1, 0] * x + matrix[1, 1] * y) / rsun
    disk = xx ** 2 + yy ** 2 <= .98 ** 2
    z = np.sqrt(np.maximum(1e-8, 1 - xx ** 2 - yy ** 2))
    b0 = np.deg2rad(h.get("SOLAR_B0", 0))
    lat = np.rad2deg(np.arcsin(np.clip(yy * np.cos(b0) + z * np.sin(b0), -1, 1)))
    lon = np.rad2deg(np.arctan2(xx, z * np.cos(b0) - yy * np.sin(b0)))
    area = np.where(disk, 1 / (z * 2 * np.pi * rsun ** 2), 0)
    return disk, lat, lon, area


def dark_regions(rad, dqf, coordinates, threshold):
    disk, lat, lon, area = coordinates
    valid = disk & (dqf == 0) & np.isfinite(rad) & (rad >= 0)
    reference = np.median(rad[valid]) if valid.any() else np.nan
    dark = valid & (rad < threshold * reference)
    structure = np.ones((3, 3), dtype=bool)
    dark = ndimage.binary_closing(ndimage.binary_opening(dark, structure), structure) & valid
    labels, count = ndimage.label(dark, structure)
    totals = np.bincount(labels.ravel(), weights=area.ravel(), minlength=count + 1)
    keep = totals >= .0005
    keep[0] = False
    mask = keep[labels]
    labels, count = ndimage.label(mask, structure)
    return mask, labels, count, valid, lat, lon, area


def dark_area_stats(rad, dqf, header):
    # 임계 0.35/0.45/0.55 어두운 영역의 면적·위치 (180열)
    rad, dqf = np.asarray(rad, dtype=float), np.asarray(dqf)
    keys = {"CRPIX1", "CRPIX2", "RSUN", "CROTA", "SOLAR_B0", "PC1_1", "PC1_2", "PC2_1", "PC2_2"}
    h = {str(k).upper(): float(np.asarray(v).item()) for k, v in header.items() if str(k).upper() in keys}
    rsun = h["RSUN"]
    if all(key in h for key in ("PC1_1", "PC1_2", "PC2_1", "PC2_2")):
        scale = np.linalg.svd(np.array([[h["PC1_1"], h["PC1_2"]], [h["PC2_1"], h["PC2_2"]]]), compute_uv=False).min()
        if scale <= 0:
            raise ValueError("singular SUVI PC matrix")
    else:
        scale = 1.
    radius = rsun / scale + 2       # 디스크를 덮는 정사각형으로 자름 (재표본 없음, 좌표계 유지)
    cx, cy = h["CRPIX1"] - 1, h["CRPIX2"] - 1
    x0, x1 = max(0, int(np.floor(cx - radius))), min(rad.shape[1], int(np.ceil(cx + radius)) + 1)
    y0, y1 = max(0, int(np.floor(cy - radius))), min(rad.shape[0], int(np.ceil(cy + radius)) + 1)
    rad, dqf = rad[y0:y1, x0:x1], dqf[y0:y1, x0:x1]
    h["CRPIX1"], h["CRPIX2"] = cx + 1 - x0, cy + 1 - y0
    coordinates = dark_geometry(rad.shape, h)
    out = {}
    for threshold in THRESHOLDS:
        mask, labels, count, valid, lat, lon, area = dark_regions(rad, dqf, coordinates, threshold)
        result = {"area": float(area[mask].sum()), "count": float(count),
                  "valid_fraction": float(valid.sum() / max(1, (area > 0).sum()))}
        objects = []
        for label, bounds in enumerate(ndimage.find_objects(labels), 1):
            if bounds is None:
                continue
            inside = labels[bounds] == label
            weights = area[bounds][inside]
            la, lo = lat[bounds][inside], lon[bounds][inside]
            objects.append((float(weights.sum()), float(np.average(la, weights=weights)),
                            float(np.average(lo, weights=weights)), float(np.ptp(lo)), float(np.ptp(la))))
        objects.sort(key=lambda row: (-row[0], row[1], row[2]))
        result["max_area"] = objects[0][0] if objects else 0.
        for i in range(3):
            values = objects[i] if i < len(objects) else (0.,) * 5
            result.update({f"top{i + 1}_{name}": value for name, value in
                           zip(("area", "lat", "lon", "lon_width", "lat_width"), values)})
        for width in (15, 30):
            result[f"meridian{width}_area"] = float(area[mask & (np.abs(lon) <= width)].sum())
            result[f"meridian{width}_equatorial_area"] = float(area[mask & (np.abs(lon) <= width) & (np.abs(lat) <= 30)].sum())
        result["equatorial_area"] = float(area[mask & (np.abs(lat) <= 30)].sum())
        longitude = np.clip(((lon + 90) / 15).astype(int), 0, 11)
        latitude = np.clip(((lat + 90) / 60).astype(int), 0, 2)
        totals = np.bincount((longitude * 3 + latitude)[mask], weights=area[mask], minlength=36)
        result.update({f"lon{i:02d}_lat{j}_area": float(totals[i * 3 + j]) for i in range(12) for j in range(3)})
        if not valid.any():
            result = {key: np.nan for key in result}
        out.update({f"suvi_t{round(threshold * 100):03d}_{key}": value for key, value in result.items()})
    return out


def extract_one(path):
    try:
        rad, dqf, header = read_suvi_frame(path)
        res = {**cell_stats(rad, dqf, header), **dark_area_stats(rad, dqf, header)}
        return path, (tuple(res), np.fromiter(res.values(), dtype=np.float64, count=len(res))), None
    except Exception as exc:   # 읽지 못한 파일: 그 칸은 NaN, 목록은 MANIFEST에
        return path, None, f"{type(exc).__name__}: {exc}"


files = []
for p in sorted(NC_DIR.glob("*/*/*.nc")):
    m = SUVI_NAME.search(p.name)
    if m:
        files.append((str(p), stamp(m.groups()[6:])))
nc = pd.DataFrame(files, columns=["path", "obs_end"])
nc["bin"] = bin_hour(nc["obs_end"])
nc = nc.sort_values(["bin", "obs_end"])
multi = int(nc["bin"].duplicated().sum())
nc = nc.drop_duplicates("bin", keep="last")
nc = nc[nc["bin"].isin(all_origins)].reset_index(drop=True)
print(f"NetCDF {len(files):,}개 | 같은 칸의 중복 {multi} | origin 칸에 해당 {len(nc):,}개 | {nc.obs_end.min()} ~ {nc.obs_end.max()}")

t0 = time.time()
rows, errors = {}, {}
with ProcessPoolExecutor(WORKERS, mp_context=multiprocessing.get_context("fork")) as ex:
    for i, (path, res, err) in enumerate(ex.map(extract_one, nc["path"], chunksize=16), 1):
        if err:
            errors[path] = err
        else:
            rows[path] = res
        if i % 2000 == 0 or i == len(nc):
            print(f"  {i:,}/{len(nc):,} ({time.time() - t0:.0f}s)", flush=True)
cols = list(next(iter(rows.values()))[0])
assert all(k == tuple(cols) for k, _ in rows.values())
empty = np.full(len(cols), np.nan)
frame = pd.DataFrame(np.stack([rows[p][1] if p in rows else empty for p in nc["path"]]), columns=cols,
                     index=pd.DatetimeIndex(nc["bin"]))
frame.insert(0, "obs_end", pd.DatetimeIndex(nc["obs_end"]))
print(f"추출 {len(rows):,}개, 실패 {len(errors)}개, {time.time() - t0:.0f}s | 열 {len(cols)} (셀 통계 {sum(c.startswith('suvi_disk') or c.startswith('suvi_lon') for c in cols)}, 면적 {sum(c.startswith('suvi_t0') for c in cols)})")
for p, e in list(errors.items())[:10]:
    print("  실패:", p, e)

NetCDF 33,842개 | 같은 칸의 중복 0 | origin 칸에 해당 32,987개 | 2022-11-22 21:58:30.600000+00:00 ~ 2026-09-26 22:59:15.100000+00:00


  2,000/32,987 (700s)


  4,000/32,987 (1066s)


  6,000/32,987 (1460s)


  8,000/32,987 (1839s)


  10,000/32,987 (2202s)


... (9 outputs left out)

  30,000/32,987 (6054s)


  32,000/32,987 (6393s)


  32,987/32,987 (6542s)


추출 32,983개, 실패 4개, 6548s | 열 328 (셀 통계 148, 면적 180)
  실패: /home/t-lab01/sunrun/data-v2/raw/SUVI/nc/g18/2025/340/OR_SUVI-L1b-Fe195_G18_s20253400856275_e20253400856285_c20253400859414.nc ValueError: RSUN must be positive
  실패: /home/t-lab01/sunrun/data-v2/raw/SUVI/nc/g18/2025/340/OR_SUVI-L1b-Fe195_G18_s20253401056279_e20253401056289_c20253401059409.nc ValueError: RSUN must be positive
  실패: /home/t-lab01/sunrun/data-v2/raw/SUVI/nc/g18/2025/340/OR_SUVI-L1b-Fe195_G18_s20253401256282_e20253401256292_c20253401259429.nc ValueError: RSUN must be positive
  실패: /home/t-lab01/sunrun/data-v2/raw/SUVI/nc/g18/2025/340/OR_SUVI-L1b-Fe195_G18_s20253401456286_e20253401456296_c20253401459433.nc ValueError: RSUN must be positive


## 4. SUVI 피처

- origin O는 **칸 O의 프레임만** 씁니다(`obs_end` ∈ [O−1h, O)). 칸 O에 프레임이 없으면 SUVI 열이 모두 NaN이고, 이전 프레임으로 되돌아가지 않습니다.
- `img_age_h = O − obs_end`(시간). 칸 규칙 때문에 0 초과 1 이하입니다.
- **strip 15열** `strip_v{350,450,550,650,750}_{logmed,dark,cover}`는 h마다 다릅니다. 속도 v의 태양풍이 target 시각 O+h에 L1에 닿으려면 떠난 경도가 λ(h, v) = 0.55°/h × (τ_v − (h + img_age_h))이고, τ_v = (1 AU − 2.5 R☉)/v입니다. 그 경도를 중심으로 폭 10° strip이 위도 가운데 칸(`lat1`) 경도 12칸과 겹치는 비율로 `log_median`·`dark_fraction`·`valid_fraction`을 가중 평균합니다. 계산식은 `d1_longitude_strips/strips.py`에서 옮겼고, cover < 0.5 마스크와 6h stale 마스크는 뺐습니다. strip이 디스크 밖(겹침 0)이면 NaN입니다.

In [5]:
OMEGA, D_KM, CELL_W, STRIP_HALF = 0.55, 147858620.7, 15.0, 5.0
SPEEDS = (350, 450, 550, 650, 750)
TAU = np.array([D_KM / v / 3600.0 for v in SPEEDS])
STRIP_STATS = {"logmed": "log_median", "dark": "dark_fraction", "cover": "valid_fraction"}
STRIP_COLS = [f"strip_v{v}_{s}" for v in SPEEDS for s in STRIP_STATS]
H_ALL = np.arange(1, H + 1)
EDGES = -90.0 + CELL_W * np.arange(13)

suvi_feat = frame.reindex(all_origins)
img_age_h = ((all_origins - pd.DatetimeIndex(suvi_feat["obs_end"])) / HOUR).to_numpy(dtype=float)
suvi_feat = suvi_feat.drop(columns="obs_end")
suvi_feat.insert(0, "img_age_h", img_age_h)
LAT1 = {s: suvi_feat[[f"suvi_lon{i:02d}_lat1_{suffix}" for i in range(12)]].to_numpy(float) for s, suffix in STRIP_STATS.items()}


def strip_block(idx):
    # origin 위치 idx (n,) -> (n*72, 15) strip 값
    age = img_age_h[idx]
    lam = OMEGA * (TAU[None, None, :] - (H_ALL[None, :, None] + age[:, None, None]))           # (n, 72, 5)
    w = np.clip(np.minimum(lam[..., None] + STRIP_HALF, EDGES[1:]) - np.maximum(lam[..., None] - STRIP_HALF, EDGES[:-1]),
                0, None) / (2 * STRIP_HALF)                                                    # (n, 72, 5, 12)
    ws = w.sum(-1)
    out = np.empty((len(idx), H, len(SPEEDS), 3))
    for k, s in enumerate(STRIP_STATS):
        num = (w * LAT1[s][idx][:, None, None, :]).sum(-1)
        out[..., k] = np.where(ws > 0, num / np.where(ws > 0, ws, 1), np.nan)
    return out.reshape(len(idx) * H, len(STRIP_COLS))


have = np.isfinite(img_age_h)
print(f"SUVI 프레임이 있는 origin: {have.sum():,}/{len(all_origins):,} ({have.mean():.1%}) | img_age_h 범위 {np.nanmin(img_age_h):.3f}~{np.nanmax(img_age_h):.3f}h")

SUVI 프레임이 있는 origin: 32,987/34,968 (94.3%) | img_age_h 범위 0.006~0.998h


## 5. ENLIL 피처

`raw/ENLIL/donki/`의 WSA-ENLIL 실행 중 `estimatedShockArrivalTime`이 있는 것만 씁니다. 실행 하나 = (`modelCompletionTime`, 도착 예상 시각, max(kp_90, kp_135, kp_180), `isEarthGB`)이고, 같은 내용의 실행은 하나로 칩니다.
- 실행은 칸 `bin_hour(modelCompletionTime)`에 들어가고, origin O는 칸이 [O−7일, O]인 실행을 씁니다.
- dt = 도착 예상 − O(시간). 앞으로 올 것 = dt ≥ −6h, 지난 것 = −96h ≤ dt < −6h.
- 이 기준으로 다음 열을 만듭니다.
  - `enlil_next_arr_h`: 앞으로 올 것의 최소 dt
  - `enlil_since_arr_h`: 지난 것의 −최대 dt
  - `enlil_n_pending`: 앞으로 올 것의 개수
  - `enlil_kp`: 앞으로 올 것의 최대 kp, 없으면 0
  - `enlil_glancing`: 앞으로 올 것이 모두 glancing blow인지
- h마다 다른 열은 `enlil_dt_target = h − enlil_next_arr_h`입니다.

In [6]:
runs = []
for p in sorted(ENLIL_DIR.glob("WSAEnlilSimulations_*.json")):
    for s in json.loads(p.read_text()):
        if s.get("estimatedShockArrivalTime"):
            runs.append((pd.Timestamp(s["modelCompletionTime"]), pd.Timestamp(s["estimatedShockArrivalTime"]),
                         max(s.get("kp_90") or 0, s.get("kp_135") or 0, s.get("kp_180") or 0), bool(s.get("isEarthGB"))))
runs = pd.DataFrame(runs, columns=["done", "arrival", "kp", "glancing"]).drop_duplicates()
runs["bin"] = bin_hour(runs["done"])
runs = runs.sort_values("bin").reset_index(drop=True)

rb = runs["bin"].to_numpy("datetime64[ns]").astype(np.int64)
ra = pd.DatetimeIndex(runs["arrival"]).tz_convert("UTC").to_numpy("datetime64[ns]").astype(np.int64)
rk, rg = runs["kp"].to_numpy(float), runs["glancing"].to_numpy(bool)
on = all_origins.to_numpy("datetime64[ns]").astype(np.int64)
NS_H, WEEK = 3600 * 10 ** 9, 7 * 24 * 3600 * 10 ** 9
enl = np.full((len(on), 5), np.nan)
hi = np.searchsorted(rb, on, side="right")
lo = np.searchsorted(rb, on - WEEK, side="left")
for i in range(len(on)):
    dt = (ra[lo[i]:hi[i]] - on[i]) / NS_H
    fut, past = dt >= -6, (dt < -6) & (dt >= -96)
    enl[i] = [dt[fut].min() if fut.any() else np.nan, -dt[past].max() if past.any() else np.nan, fut.sum(),
              rk[lo[i]:hi[i]][fut].max() if fut.any() else 0, float(rg[lo[i]:hi[i]][fut].all()) if fut.any() else np.nan]
ENLIL_COLS = ["enlil_next_arr_h", "enlil_since_arr_h", "enlil_n_pending", "enlil_kp", "enlil_glancing"]
enlil_feat = pd.DataFrame(enl, index=all_origins, columns=ENLIL_COLS)
print(f"ENLIL 실행 {len(runs):,}개 ({runs.done.min()} ~ {runs.done.max()}) | 앞으로 올 실행이 있는 origin {(enlil_feat.enlil_n_pending > 0).mean():.1%}")

ENLIL 실행 964개 (2022-07-04 20:17:00+00:00 ~ 2026-10-21 15:31:00+00:00) | 앞으로 올 실행이 있는 origin 57.5%


## 6. df 조립·저장

- 형식은 예전 df와 같은 (`set`, `origin`, `h`) 긴 형식이고, origin 하나가 h = 1…72의 72행입니다.
  - `set = train`: origin 2022-10-01T00Z ~ 2026-06-23T00Z(F3 cutoff − 144h), 1h 간격.
  - `set = query`: origin 2025-12-31T23Z ~ 2026-09-26T23Z, 1h 간격, 6,457개. `period`는 origin이 속한 F1/F2/F3이고, 기간 사이 71개 origin은 비어 있습니다.
  - 같은 origin이 두 set에 모두 있으면 값은 같고 `set`만 다릅니다.
- `target_time = origin + h`. `y`는 1단계의 T 라벨 status 0 속도입니다.
- `fit_P = (set == train) & (origin + 144h ≤ cutoff_P)`, cutoff는 F1 2025-12-29, F2 2026-03-29, F3 2026-06-29입니다.
- 열 순서는 메타 9열, 상수 피처(ACE, SUVI, ENLIL), h마다 다른 피처(`rec26/27/28`, `rec_mean`, strip 15열, `enlil_dt_target`)입니다.
  - `rec{d}`는 ACE 속도 칸(target_time − d일)이고, d = 26/27/28은 27일 태양 자전 재귀입니다. h ≤ 72이므로 그 칸은 항상 origin 이전입니다.
  - `rec_mean`은 셋 중 NaN이 아닌 값의 평균입니다.
- 메모리를 아끼려고 origin 1,000개씩 펼쳐 row group으로 이어 씁니다(zstd). 숫자 피처는 float32, `y`는 float64입니다.

In [7]:
const = pd.concat([ace_feat, suvi_feat, enlil_feat], axis=1)
assert const.index.equals(all_origins) and const.columns.is_unique
CONST_COLS = list(const.columns)
VARY_COLS = ["rec26", "rec27", "rec28", "rec_mean"] + STRIP_COLS + ["enlil_dt_target"]
CONST = const.to_numpy(np.float32)
pos = pd.Series(np.arange(len(all_origins)), index=all_origins)
speed_bin = ace_bin["ace_speed_kms"]
next_arr = enlil_feat["enlil_next_arr_h"].to_numpy(float)

SET_DICT = pa.array(["train", "query"])
PERIOD_DICT = pa.array(list(PERIODS))
period_of = pd.Series(pd.NA, index=query_origins, dtype="object")
for name, (first, last, _) in PERIODS.items():
    period_of[(query_origins >= pd.Timestamp(first)) & (query_origins <= pd.Timestamp(last))] = name
CUTOFF = {p: pd.Timestamp(c) for p, (_, _, c) in PERIODS.items()}


def dict_col(values, dictionary):
    lookup = {v: i for i, v in enumerate(dictionary.to_pylist())}
    idx = pa.array([lookup.get(v) if isinstance(v, str) else None for v in values], type=pa.int8())
    return pa.DictionaryArray.from_arrays(idx, dictionary)


def block(set_name, origins):
    n = len(origins)
    idx = pos[origins].to_numpy()
    o = np.repeat(origins.to_numpy("datetime64[ns]"), H)
    h = np.tile(H_ALL, n).astype(np.int16)
    tt = o + h.astype("timedelta64[h]").astype("timedelta64[ns]")
    ttd = pd.DatetimeIndex(tt, tz="UTC")
    od = pd.DatetimeIndex(o, tz="UTC")
    periods = np.repeat(period_of.reindex(origins).to_numpy() if set_name == "query" else np.full(n, None), H)
    cols = {"set": dict_col([set_name] * (n * H), SET_DICT),
            "origin": pa.array(od, type=pa.timestamp("ns", tz="UTC")),
            "h": pa.array(h),
            "target_time": pa.array(ttd, type=pa.timestamp("ns", tz="UTC")),
            "period": dict_col(periods, PERIOD_DICT)}
    for p, cut in CUTOFF.items():
        cols[f"fit_{p}"] = pa.array((set_name == "train") & (od + pd.Timedelta(hours=MASK_HOURS) <= cut))
    cols["y"] = pa.array(y_series.reindex(ttd).to_numpy(float))
    c = np.repeat(CONST[idx], H, axis=0)
    for j, name in enumerate(CONST_COLS):
        cols[name] = pa.array(c[:, j])
    rec = np.stack([speed_bin.reindex(ttd - pd.Timedelta(days=d)).to_numpy(float) for d in (26, 27, 28)], 1)
    vary = [rec[:, 0], rec[:, 1], rec[:, 2], np.nanmean(rec, 1)]
    st = strip_block(idx)
    vary += [st[:, k] for k in range(st.shape[1])]
    vary.append(h - np.repeat(next_arr[idx], H))
    for name, val in zip(VARY_COLS, vary):
        cols[name] = pa.array(np.asarray(val, dtype=np.float32))
    return pa.table(cols)


OUT.mkdir(parents=True, exist_ok=True)
path, tmp = OUT / "df.base.parquet", OUT / "df.base.parquet.tmp"   # 6b절이 20열을 붙여 df.parquet로 씀
t0, writer, n_rows = time.time(), None, {}
try:
    for set_name, origins in (("train", train_origins), ("query", query_origins)):
        n_rows[set_name] = 0
        for s in range(0, len(origins), 1000):
            tbl = block(set_name, origins[s:s + 1000])
            if writer is None:
                writer = pq.ParquetWriter(tmp, tbl.schema, compression="zstd")
            writer.write_table(tbl)
            n_rows[set_name] += tbl.num_rows
        print(f"  {set_name}: {n_rows[set_name]:,}행 ({time.time() - t0:.0f}s)", flush=True)
finally:
    if writer is not None:
        writer.close()
tmp.replace(path)
columns = tbl.schema.names
print(f"{path}: {sum(n_rows.values()):,}행 × {len(columns)}열, {path.stat().st_size / 1e9:.2f} GB, {time.time() - t0:.0f}s")

  train: 2,351,880행 (30s)


  query: 464,904행 (36s)


/home/t-lab01/sunrun/data-v2/input/df.parquet: 2,816,784행 × 446열, 0.21 GB, 36s


## 6b. 추가 피처 (원문 이벤트·보조 원천)

6절이 쓴 446열 표(`df.base.parquet`)를 읽어 20열을 더한 뒤 `df.parquet`로 씁니다. 원천은 `fetch.py`의 `donki`, `srs`,
`discussion`, `ace-daily`, `wsa`입니다.

- 모든 값은 origin 이전에 공개된 원문만 씁니다(공개 시각 = DONKI 본문 발행 시각, SRS `:Issued:`, discussion `Last-Modified`,
  ACE 일별 파일 `:Created:`, WSA S3 `LastModified`).
- 결측 표시값 `-999`는 피처 정의의 일부입니다(앞값 채움 대상이 아님). 그 밖의 처리는 하지 않습니다.
- `onset700_hazard`만 학습 정답을 씁니다. 보조 분류기는 2025-09-23(F1 첫 cutoff의 3개월 전)까지 관측된 학습 y로만 적합합니다.

| 열 | 정의 |
|---|---|
| `pub_enlil_next_arr_h`, `pub_enlil_since_arr_h`, `pub_enlil_glancing` | 살아 있는 DONKI 지구 도착 예보까지 남은 시간, 6–96 h 전 예보 이후 시간, glancing 여부 |
| `pub_cme_next_{speed_kms,half_angle_deg,lon_deg,lat_deg}`, `pub_cme_last_speed_kms` | 다음 예보 CME의 속도·반각·경위도, 지난 예보 CME 속도 |
| `ace_log_proton_entropy` | log10 T − 5 − (2/3) log10 n |
| `ace_nb_cocompression_6h` | 최근 6 h의 min(log2 n, log2 Bt) 상승(직전 24 h 중앙값 대비)의 중앙값 |
| `swpc_hss_coverage49_72` | 최신 SWPC 예보 토의(18 h 이내)가 고속류를 예보한 날짜에 h49–72가 걸리는 비율 |
| `ace_proton_ram_pressure_npa` | 1.6726e-6 n v² (h별 앞값 채움 값) |
| `srs_earthward_eruptive_area` | 최신 SRS(36 h 이내)의 지구 쪽 분출형 흑점군 면적 log1p(합/100) |
| `cme_phase_rec27` | 다음 DONKI 도착 시각 근처 가우시안 가중 × (rec27 − v_lag0) |
| `suvi_t055_lowlat_top1_lon_width_cap67_deg` | 저위도 최대 어두운 영역 경도 폭 min(w, 67) |
| `suvi_ballistic_dark_contrast` | h49–72의 strip_v550_dark − strip_v450_dark 평균 |
| `rec27_ramp`, `rec54_consensus` | ACE 1분 일별 파일로 만든 27일 전 속도 3 h 중앙값 차이, 27·54일 전 속도의 최솟값 |
| `enlil_earth_speed` | 48 h 안에 공개된 최신 WSA-Enlil 실행(NOAA open-data S3 `noaa-wsa-enlil-pds`만)의 지구 위치 예보 속도(target_time 칸) |
| `onset700_hazard` | 700 km/s 첫 도달 확률(분기마다 다시 적합한 보조 XGBoost, 2025-09-23 freeze) |

In [ ]:
import email.utils
from itertools import combinations

import xgboost as xgb
from scipy.io import netcdf_file

BASE = OUT / "df.base.parquet"          # 6절이 쓰는 446열 표
EXT_MISSING = -999.
DONKI_DIR = DATA / "raw/DONKI/notifications"
SRS_DIR = DATA / "raw/NOAA/srs"
DISCUSSION_DIR = DATA / "raw/NOAA/discussion"
ACE_DAILY_DIR = DATA / "raw/ACE/daily-txt/swepam"
WSA_DIR = DATA / "raw/ENLIL/wsa-s3"
EXT_ORDER = ["pub_enlil_next_arr_h", "pub_enlil_since_arr_h", "pub_enlil_glancing", "pub_cme_next_speed_kms",
             "pub_cme_next_half_angle_deg", "pub_cme_next_lon_deg", "pub_cme_next_lat_deg", "pub_cme_last_speed_kms",
             "ace_log_proton_entropy", "ace_nb_cocompression_6h", "swpc_hss_coverage49_72",
             "ace_proton_ram_pressure_npa", "srs_earthward_eruptive_area", "cme_phase_rec27",
             "suvi_t055_lowlat_top1_lon_width_cap67_deg", "suvi_ballistic_dark_contrast", "rec27_ramp",
             "enlil_earth_speed", "onset700_hazard", "rec54_consensus"]
ext_index = pd.DatetimeIndex(all_origins)
N_EXT = len(ext_index)
EXT_CONST, EXT_VARY = {}, {}             # 이름 -> (origin,) 또는 (origin, 72) float32


def first_rows(columns):
    """h = 1 행의 채우지 않은 값, origin당 1행 (train·query 사본이 같아야 함)."""
    t = pd.read_parquet(BASE, columns=["origin", *columns], filters=[("h", "=", 1)])
    per = t.groupby("origin")[columns]
    if (per.nunique(dropna=False) > 1).any().any():
        raise ValueError("conflicting duplicate origin values")
    out = per.first()
    out.index = pd.DatetimeIndex(out.index)
    assert out.index.equals(ext_index)
    return out


def lead_grid(column):
    """채우지 않은 (origin, h) 값을 (origin, 72) 행렬로."""
    t = pd.read_parquet(BASE, columns=["origin", "h", column])
    u = t.drop_duplicates(["origin", "h", column])          # NaN끼리는 같은 값으로 봄
    if u.duplicated(["origin", "h"]).any():
        raise ValueError(f"conflicting duplicate (origin, h) values in {column}")
    u = u.sort_values(["origin", "h"], kind="stable")
    assert len(u) == N_EXT * H and pd.DatetimeIndex(u["origin"].iloc[::H]).equals(ext_index)
    return u[column].to_numpy().reshape(N_EXT, H)

In [ ]:
# DONKI CME 알림: 본문 발행 시각 <= origin인 지구 도착 예보·철회만 (research/_module/public_events.py와 같은 문법)
PUB_T = ["pub_enlil_next_arr_h", "pub_enlil_since_arr_h", "pub_enlil_glancing"]
PUB_K = ["pub_cme_next_speed_kms", "pub_cme_next_half_angle_deg", "pub_cme_next_lon_deg", "pub_cme_next_lat_deg",
         "pub_cme_last_speed_kms"]
PUB_FEATURES = PUB_T + PUB_K
D_TIME = r"\d{4}-\d{2}-\d{2}T\d{2}:\d{2}(?::\d{2})?Z"
D_CME_ID = r"\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}-CME-\d+"
D_MESSAGE_ID = r"\d{8}-AL-\d+"
D_EARTH = r"NASA missions near Earth"
D_ARRIVAL = re.compile(
    rf"(?P<modal>(?:will|may|might)\s+|(?:is|are)\s+estimated\s+to\s+)"
    rf"(?:possibly\s+)?(?:reach|impact)\s+{D_EARTH}\s+at\s+(?:about\s+)?(?P<arrival>{D_TIME})", re.I)
D_NEGATIVE = re.compile(
    rf"(?:will\s+not|is\s+not\s+expected\s+to|are\s+not\s+expected\s+to|"
    rf"is\s+no\s+longer\s+expected\s+to|are\s+no\s+longer\s+expected\s+to)"
    rf"\s+(?:reach|impact|affect)\s+{D_EARTH}", re.I)
D_KINEMATICS = re.compile(
    r"Estimated speed:\s*~(\d+) km/s\.\s*"
    r"Estimated opening half-angle:\s*(\d+) deg\.\s*"
    r"Direction \(lon\./lat\.\):\s*(-?\d+)/(-?\d+) in Heliocentric Earth Equatorial coordinates\.\s*"
    rf"Activity ID:\s*({D_CME_ID})")


def parse_message(message):
    """추적 가능한 예보/철회 또는 명시적 거절 기록."""
    body = message.get("messageBody", "")
    record = dict(message_id=message.get("messageID"), message_url=message.get("messageURL"),
                  api_issue=message.get("messageIssueTime"), status="rejected", reason="",
                  body_issue=None, event_ids=[], previous_ids=[], arrival=None, glancing=-1,
                  earth_text="", action=None)
    issue_match = re.search(r"^##\s*Message Issue Date:\s*(\S+)", body, re.M | re.I)
    id_match = re.search(r"^##\s*Message ID:\s*(\S+)", body, re.M | re.I)
    if not issue_match or not id_match:
        record["reason"] = "missing_body_identity_or_issue"
        return record
    api_issue = pd.to_datetime(record["api_issue"], utc=True, errors="coerce")
    issue = pd.to_datetime(issue_match.group(1), utc=True, errors="coerce")
    record["body_issue"] = None if pd.isna(issue) else issue.isoformat()
    if id_match.group(1) != record["message_id"]:
        record["reason"] = "body_message_id_mismatch"
        return record
    if pd.isna(issue) or pd.isna(api_issue) or not (api_issue <= issue < api_issue + pd.Timedelta(minutes=1)):
        record["reason"] = "body_issue_not_within_api_issue_minute"
        return record
    section = re.split(r"##\s*Summary\s*:", body, maxsplit=1, flags=re.I)
    if len(section) != 2:
        record["reason"] = "missing_summary"
        return record
    summary = re.split(r"##\s*Notes\s*:", section[1], maxsplit=1, flags=re.I)[0]
    if re.search(r"\bensemble\b", body[:body.find("## Summary:")] + summary, re.I):
        record["reason"] = "ensemble_excluded"
        return record
    summary = re.split(r"Links? to (?:the )?(?:movies|simulations|modeled|modelled)", summary,
                       maxsplit=1, flags=re.I)[0]
    flat = re.sub(r"\s+", " ", summary).strip()
    record["previous_ids"] = sorted(set(re.findall(D_MESSAGE_ID, flat)))
    all_ids = sorted(set(re.findall(D_CME_ID, flat)))
    paragraphs = [re.sub(r"\s+", " ", p).strip() for p in re.split(r"\n\s*\n", summary)]
    positive = [(p, m) for p in paragraphs for m in D_ARRIVAL.finditer(p)]
    negative = [p for p in paragraphs if D_NEGATIVE.search(p)]
    if positive and negative:
        record["reason"] = "conflicting_earth_actions"
        return record
    if negative:
        earth_text = " ".join(negative)
        action, arrival, glancing = "retract", None, -1
    elif positive:
        arrival_values = {m.group("arrival") for _, m in positive}
        if len(arrival_values) != 1:
            record["reason"] = "multiple_earth_arrivals_ambiguous"
            record["earth_text"] = " ".join(p for p, _ in positive)
            return record
        earth_text = " ".join(dict.fromkeys(p for p, _ in positive))
        action, arrival = "forecast", pd.Timestamp(next(iter(arrival_values)))
        if arrival <= issue:
            record["reason"] = "arrival_not_after_public_issue"
            record["earth_text"] = earth_text
            return record
        geometries = set()
        for paragraph, match in positive:
            prefix = paragraph[:match.start()]
            direct = re.search(r"(leading edge|flank)\s+of\s+(?:the\s+)?CMEs?\s*$", prefix, re.I)
            if direct:
                geometries.add(0 if direct.group(1).lower() == "leading edge" else 1)
            if re.search(rf"glancing blow\s+(?:at|to)\s+{D_EARTH}", paragraph, re.I):
                geometries.add(1)
        glancing = next(iter(geometries)) if len(geometries) == 1 else -1
    else:
        record["reason"] = "no_explicit_deterministic_earth_arrival_or_retraction"
        return record
    record["earth_text"] = earth_text
    earth_ids = sorted(set(re.findall(D_CME_ID, earth_text)))
    if earth_ids:
        event_ids = earth_ids
    elif len(all_ids) == 1:
        event_ids = all_ids
    elif len(all_ids) > 1 and re.search(r"\bCMEs\b", earth_text):
        event_ids = all_ids
    elif len(all_ids) > 1:
        dates = set(re.findall(D_TIME, earth_text))
        event_ids = [identifier for identifier in all_ids if identifier.split("-CME-")[0][:16] + "Z" in dates]
        if len(event_ids) != 1:
            record["reason"] = "ambiguous_earth_event_identity"
            return record
    else:
        event_ids = []
        if not record["previous_ids"]:
            record["reason"] = "missing_earth_event_identity"
            return record
    record.update(status="accepted", reason="explicit_public_earth_" + action,
                  event_ids=event_ids, action=action,
                  arrival=None if arrival is None else arrival.isoformat(), glancing=glancing)
    return record


def parse_corpus(messages):
    parsed = [parse_message(m) for m in messages]
    parsed.sort(key=lambda r: (r["body_issue"] or "", r["message_id"]))
    earlier = {}
    for record in parsed:
        if record["status"] != "accepted":
            continue
        if not record["event_ids"]:
            previous = [earlier[k] for k in record["previous_ids"] if k in earlier]
            groups = {tuple(r["event_ids"]) for r in previous}
            if len(groups) != 1 or len(previous) != len(record["previous_ids"]):
                record.update(status="rejected", reason="ambiguous_or_unavailable_previous_identity")
                continue
            record["event_ids"] = list(next(iter(groups)))
        earlier[record["message_id"]] = record
    return parsed


def kinematics(body, event_ids):
    """한 알림의 지구 예보 이벤트 중 가장 빠른 CME, 없으면 None."""
    parts = re.split(r"##\s*Summary\s*:", body, maxsplit=1, flags=re.I)
    if len(parts) != 2:
        return None
    summary = re.split(r"##\s*Notes\s*:", parts[1], maxsplit=1, flags=re.I)[0]
    blocks = [dict(speed=float(a), half_angle=float(b), lon=float(c), lat=float(d), cme=e)
              for a, b, c, d, e in D_KINEMATICS.findall(summary)]
    if (not blocks or len(blocks) != len(re.findall(r"Estimated speed", summary))
            or len({b["cme"] for b in blocks}) != len(blocks)):
        return None
    chosen = [b for b in blocks if b["cme"] in set(event_ids)] or blocks
    return max(chosen, key=lambda b: (b["speed"], b["cme"]))


def pub_features(messages, origins):
    """origin마다 T(도착 예보 상태)와 K(CME 운동학). origin 이후 발행된 알림은 쓰지 않음."""
    bodies = {m["messageID"]: m.get("messageBody", "") for m in messages}
    records = [r for r in parse_corpus(messages) if r["status"] == "accepted"]
    records.sort(key=lambda r: (pd.Timestamp(r["body_issue"]), r["message_id"]))
    speed = {r["message_id"]: kinematics(bodies[r["message_id"]], r["event_ids"])
             for r in records if r["action"] == "forecast"}
    out = np.full((len(origins), len(PUB_FEATURES)), EXT_MISSING, np.float32)
    out[:, 2] = -1.
    active, position = {}, 0
    for i, origin in enumerate(origins):
        while position < len(records) and pd.Timestamp(records[position]["body_issue"]) <= origin:
            new = records[position]
            new_ids = set(new["event_ids"])
            for key, old in list(active.items()):
                if new_ids.intersection(old["event_ids"]):
                    del active[key]
            if new["action"] == "forecast":
                active[new["message_id"]] = new
            position += 1
        for key, old in list(active.items()):
            if pd.Timestamp(old["body_issue"]) < origin - pd.Timedelta(days=7):
                del active[key]
        values = [((pd.Timestamp(r["arrival"]) - origin) / pd.Timedelta(hours=1), r) for r in active.values()]
        pending = sorted((x for x in values if x[0] >= -6), key=lambda x: (x[0], x[1]["message_id"]))
        past = sorted((x for x in values if -96 <= x[0] < -6), key=lambda x: (-x[0], x[1]["message_id"]))
        if pending:
            hours, record = pending[0]
            out[i, 0], out[i, 2] = hours, record["glancing"]
            cme = speed[record["message_id"]]
            if cme:
                out[i, 3:7] = cme["speed"], cme["half_angle"], cme["lon"], cme["lat"]
        if past:
            hours, record = past[0]
            out[i, 1] = -hours
            cme = speed[record["message_id"]]
            if cme:
                out[i, 7] = cme["speed"]
    return out


donki_by_id = {}
for p in sorted(DONKI_DIR.glob("notifications_*.json")):
    for message in json.loads(p.read_text()):
        donki_by_id[message.get("messageID")] = message
donki_messages = list(donki_by_id.values())
pub = pub_features(donki_messages, ext_index)
for j, name in enumerate(PUB_FEATURES):
    EXT_CONST[name] = pub[:, j]
print(f"DONKI 알림 {len(donki_messages):,}개 | 다음 도착 예보가 있는 origin {(pub[:, 0] != EXT_MISSING).mean():.1%}")

In [ ]:
# ACE 파생 3열, NOAA SRS 지구 쪽 분출형 흑점 면적, SWPC 예보 토의의 고속류 날짜 비율
def ext_issued(text):
    match = re.search(r"^:Issued:\s*(\d{4} \w{3} \d{1,2} \d{4}) UTC", text, re.M)
    return pd.Timestamp(pd.to_datetime(match.group(1), format="%Y %b %d %H%M"), tz="UTC") if match else None


SRS_CLASSES = {"FKC", "EKC", "DKC", "FKI", "EKI", "DKI"}
SRS_LOCATION = re.compile(r"^([NS])(\d{2})([EW])(\d{2})$")


def srs_report(text, day):
    issued = ext_issued(text)
    if issued is None:
        return None
    available = max(issued, day + pd.Timedelta(minutes=40)) + HOUR
    lines = text.splitlines()
    try:
        first = next(i for i, l in enumerate(lines) if l.strip().startswith("I.") and "Regions with Sunspots" in l)
        last = next(i for i, l in enumerate(lines) if i > first and l.strip().startswith("IA."))
    except StopIteration:
        return available, np.nan
    body = [l for l in lines[first + 1:last] if l.strip()]
    if not body or body[0].split()[:4] != ["Nmbr", "Location", "Lo", "Area"]:
        return available, np.nan
    total = 0.
    for line in body[1:]:
        token = line.split()
        if token == ["None"]:
            continue
        if len(token) < 5:
            return available, np.nan
        location = SRS_LOCATION.match(token[1])
        try:
            area = float(token[3])
        except ValueError:
            return available, np.nan
        if location is None or not 0 <= area <= 10000:
            return available, np.nan
        if token[4].upper() in SRS_CLASSES and int(location.group(2)) <= 40 and int(location.group(4)) <= 45:
            total += area
    return available, float(np.log1p(total / 100.))


def srs_area(origins):
    rows = []
    for path in sorted(SRS_DIR.glob("*SRS.txt")):
        report = srs_report(path.read_text(errors="replace"), pd.Timestamp(path.name[:8], tz="UTC"))
        if report is not None:
            rows.append(report)
    reports = pd.DataFrame(rows, columns=["a", "b"]).sort_values("a", kind="stable")
    a, b = pd.DatetimeIndex(reports.a).as_unit("ns").asi8, reports.b.to_numpy(float)
    o = origins.as_unit("ns").asi8
    index = np.searchsorted(a, o, side="right") - 1
    at = np.maximum(index, 0)
    use = (index >= 0) & ((o - a[at]) / 3.6e12 <= 36) & np.isfinite(b[at])
    return np.where(use, b[at], EXT_MISSING).astype(np.float32), len(rows)


MONTH_NUM = {m: i for i, m in enumerate("jan feb mar apr may jun jul aug sep oct nov dec".split(), 1)}
_MON = "(" + "|".join(MONTH_NUM) + r")\w*"
HSS_DATE = re.compile(rf"\b(\d{{1,2}})\s+{_MON}\s*-\s*(\d{{1,2}})\s+{_MON}\b"
                      rf"|\b(\d{{1,2}})\s*-\s*(\d{{1,2}})\s+{_MON}\b"
                      rf"|\bthrough\s+(\d{{1,2}})\s+{_MON}\b"
                      rf"|\b(\d{{1,2}})\s+{_MON}\b")
HSS = re.compile(r"\bhss\b|high[- ]speed streams?")
HSS_POSITIVE = re.compile(r"\b(increas|enhanc|arriv|onset\b|persist|continu|influenc|geoeffective\b)")
HSS_NEGATIVE = re.compile(r"\b(cmes?\b|transient|no\b|not\b|neither\b|unlikely\b|decreas|declin|diminish|wane|weakening\b)")


def hss_dates(text, issue):
    """고속류를 예보한 UTC 날짜 집합. 모호하면 None."""
    def resolve(day, month):
        days = (issue.floor("D") + pd.Timedelta(days=k) for k in range(-1, 4))
        hit = [d for d in days if d.day == day and d.month == month]
        if len(hit) != 1:
            raise ValueError("date")
        return hit[0]

    section = re.search(r"^Solar Wind\s*$(.*?)^Geospace\s*$", text, re.M | re.S)
    if not section or ".Forecast..." not in section.group(1):
        return None
    forecast = re.sub(r"[‐-―−]", "-", section.group(1).split(".Forecast...", 1)[1])
    forecast = re.sub(r"\s+", " ", forecast).strip()
    dates = set()
    for sentence in re.split(r"(?<=\.)\s+(?=[A-Z0-9])", forecast):
        s = sentence.lower()
        if not HSS.search(s):
            continue
        if HSS_NEGATIVE.search(s):
            return None
        if not HSS_POSITIVE.search(s):
            continue
        found = list(HSS_DATE.finditer(s))
        if len(found) != 1:
            return None
        g = found[0].groups()
        try:
            if g[0]:
                low, high = resolve(int(g[0]), MONTH_NUM[g[1]]), resolve(int(g[2]), MONTH_NUM[g[3]])
            elif g[4]:
                low, high = resolve(int(g[4]), MONTH_NUM[g[6]]), resolve(int(g[5]), MONTH_NUM[g[6]])
            elif g[7]:
                low, high = issue.floor("D"), resolve(int(g[7]), MONTH_NUM[g[8]])
            else:
                low = high = resolve(int(g[9]), MONTH_NUM[g[10]])
        except ValueError:
            return None
        if high < low:
            return None
        dates.update(pd.date_range(low, high, freq="D"))
    return dates


def hss_coverage(origins):
    rows, seen = [], {}
    for path in sorted((DISCUSSION_DIR / "txt").glob("*forecast_discussion.txt")):
        text, modified = path.read_text(errors="replace"), None
        for line in (DISCUSSION_DIR / "hdr" / path.name).read_text(errors="replace").splitlines():
            if line.lower().startswith("last-modified:"):
                modified = pd.Timestamp(email.utils.parsedate_to_datetime(line.split(":", 1)[1].strip())).tz_convert("UTC")
        issue = ext_issued(text)
        if issue is None or modified is None:
            continue
        body = re.sub(r"^:Product:.*$", "", text, flags=re.M).strip()
        if issue in seen:
            if seen[issue] != body:
                raise ValueError(f"different bodies for one issue time {issue}")
            continue
        seen[issue] = body
        rows.append((issue, max(issue, modified) + HOUR, hss_dates(text, issue)))
    reports = pd.DataFrame(rows, columns=["issue", "a", "dates"]).sort_values("a", kind="stable").reset_index(drop=True)
    index = np.searchsorted(pd.DatetimeIndex(reports.a).as_unit("ns").asi8, origins.as_unit("ns").asi8, side="right") - 1
    out = np.full(len(origins), EXT_MISSING, np.float32)
    leads = pd.to_timedelta(np.arange(49, 73), unit="h")
    for j, (origin, i) in enumerate(zip(origins, index)):
        if i < 0:
            continue
        report = reports.iloc[i]
        if origin - report.issue > 18 * HOUR or report.dates is None:
            continue
        out[j] = np.float32(np.mean([day in report.dates for day in (origin + leads).floor("D")]))
    return out, len(rows)


def proton_entropy(t, n):
    t, n = np.asarray(t, np.float64), np.asarray(n, np.float64)
    ok = np.isfinite(t) & np.isfinite(n) & (t > 0) & (n > 0)
    out = np.full(len(t), EXT_MISSING, np.float64)
    out[ok] = np.log10(t[ok]) - 5. - (2. / 3.) * np.log10(n[ok])
    return out.astype(np.float32)


def cocompression(n, b):
    """연속 시간 계열 -> 최근 6 h의 min(log2 n, log2 Bt 상승, 직전 24 h 중앙값 대비)의 중앙값."""
    n, b = np.asarray(n, np.float64), np.asarray(b, np.float64)
    valid = np.isfinite(n) & np.isfinite(b) & (n > 0) & (b > 0)
    ln = np.where(valid, np.log2(np.where(valid, n, 1)), np.nan)
    lb = np.where(valid, np.log2(np.where(valid, b, 1)), np.nan)
    out = np.full(len(n), EXT_MISSING, np.float64)
    for i in range(29, len(n)):
        before, recent = slice(i - 29, i - 5), slice(i - 5, i + 1)
        rv, wv = valid[before], valid[recent]
        if rv.sum() < 12 or wv.sum() < 3:
            continue
        nr, br = np.median(ln[before][rv]), np.median(lb[before][rv])
        out[i] = np.median(np.minimum(ln[recent][wv] - nr, lb[recent][wv] - br))
    return out.astype(np.float32)


def ram_pressure(n, v):
    n, v = (np.asarray(x, np.float32).astype(np.float64) for x in (n, v))
    ok = np.isfinite(n) & np.isfinite(v) & (n > 0) & (v > 0)
    return np.where(ok, 1.6726e-6 * n * v ** 2, EXT_MISSING).astype(np.float32)


SUVI_WIDTH_RAW = ["suvi_t055_top1_lon_width", "suvi_t055_top1_lon", "suvi_t055_top1_lat", "suvi_t055_top1_area",
                  "suvi_t055_valid_fraction", "img_age_h"]
one = first_rows(["ace_density_cm3_last", "ace_bt_nt_last", "ace_temperature_k_last", "ace_speed_kms_last", "v_lag0",
                  *SUVI_WIDTH_RAW])
EXT_CONST["ace_log_proton_entropy"] = proton_entropy(one.ace_temperature_k_last, one.ace_density_cm3_last)
hourly = pd.date_range(ext_index[0], ext_index[-1], freq="h")    # 연속 시간축 (SMOKE의 train·query 사이 빈 시간은 NaN)
cc = cocompression(one.ace_density_cm3_last.reindex(hourly), one.ace_bt_nt_last.reindex(hourly))
EXT_CONST["ace_nb_cocompression_6h"] = pd.Series(cc, index=hourly).reindex(ext_index).to_numpy(np.float32)
EXT_CONST["swpc_hss_coverage49_72"], n_discussion = hss_coverage(ext_index)
# 모델 로더와 같은 h별 앞값 채움(h = 1 시간축, 앞쪽 NaN 유지) 뒤의 n, v
filled = one[["ace_density_cm3_last", "ace_speed_kms_last"]].ffill()
EXT_CONST["ace_proton_ram_pressure_npa"] = ram_pressure(filled.ace_density_cm3_last, filled.ace_speed_kms_last)
EXT_CONST["srs_earthward_eruptive_area"], n_srs = srs_area(ext_index)
print(f"SRS {n_srs:,}개, 예보 토의 {n_discussion:,}개")

In [ ]:
# SUVI 파생 2열 (채우지 않은 값)
def suvi_width(w, lon, lat, area, frac, age):
    arr = [np.asarray(x, np.float64) for x in (w, lon, lat, area, frac, age)]
    w, lon, lat, area, frac, age = arr
    ok = np.all([np.isfinite(x) for x in arr], axis=0)
    with np.errstate(invalid="ignore"):
        ok &= (w > 0) & (w <= 360) & (np.abs(lon) <= 45) & (np.abs(lat) <= 30) & (area > 0) & (frac >= .8) & (age >= 0) & (age <= 24)
    return np.where(ok, np.minimum(w, 67.), EXT_MISSING)


def dark_contrast(high, low, age):
    high, low, age = (np.asarray(x, np.float64) for x in (high, low, age))
    ok = (high >= 0) & (high <= 1) & (low >= 0) & (low <= 1)
    count = ok.sum(1)
    mean = np.where(ok, high - low, 0.).sum(1) / np.maximum(count, 1)
    return np.where((count >= 18) & (age > 0) & (age <= 1), mean, EXT_MISSING)


EXT_CONST["suvi_t055_lowlat_top1_lon_width_cap67_deg"] = suvi_width(*(one[c].to_numpy() for c in SUVI_WIDTH_RAW)).astype(np.float32)
EXT_CONST["suvi_ballistic_dark_contrast"] = dark_contrast(lead_grid("strip_v550_dark")[:, 48:], lead_grid("strip_v450_dark")[:, 48:],
                                                          one.img_age_h).astype(np.float32)

In [ ]:
# ACE 1분 일별 파일(:Created: <= origin) 기반 재귀 2열, DONKI 도착 위상 × (rec27 − v_lag0)
REC_LAG = 648


def ace_daily_hourly(h0, hours):
    """시간 칸(floor + 1 h) 평균 속도(status 0, 30분 이상)와 그 일별 파일의 Created 시각."""
    samples = np.full(hours, np.nan)
    created = np.full(hours, np.datetime64("NaT", "m"))
    files = 0
    for path in sorted(ACE_DAILY_DIR.glob("*.txt")):
        if not path.stem.isdigit():
            continue
        day = datetime.strptime(path.stem, "%Y%m%d")
        with path.open() as handle:
            first = [next(handle), next(handle)]
            if not first[1].startswith(":Created:"):
                raise ValueError(f"{path}: missing Created header")
            publication = datetime.strptime(first[1][9:].strip(), "%Y %b %d %H%M UT")
            rows = np.loadtxt(handle, comments=("#", ":"), usecols=(0, 1, 2, 3, 6, 8), ndmin=2)
        files += 1
        if rows.size == 0:
            continue
        if not (np.all(rows[:, 0] == day.year) and np.all(rows[:, 1] == day.month) and np.all(rows[:, 2] == day.day)):
            raise ValueError(f"{path}: rows from another day")
        if len(np.unique(rows[:, 3])) != len(rows):
            raise ValueError(f"{path}: duplicate minutes")
        good = (rows[:, 4] == 0) & np.isfinite(rows[:, 5]) & (rows[:, 5] > 0)
        bins = rows[good, 3].astype(int) // 100
        counts = np.bincount(bins, minlength=24)
        sums = np.bincount(bins, weights=rows[good, 5], minlength=24)
        hours_ok = np.flatnonzero(counts >= 30)
        index = int((np.datetime64(day, "h") - h0) / np.timedelta64(1, "h")) + hours_ok + 1
        keep = (index >= 0) & (index < hours)
        samples[index[keep]] = sums[hours_ok[keep]] / counts[hours_ok[keep]]
        created[index[keep]] = np.datetime64(publication, "m")
    return samples, created, files


rec_h0 = np.datetime64("2022-08-01", "h")
rec_span = int((np.datetime64(ext_index.max().tz_convert(None), "h") - rec_h0) / np.timedelta64(1, "h")) + 1
rec_samples, rec_created, rec_files = ace_daily_hourly(rec_h0, rec_span)
if rec_files == 0:
    raise ValueError("No Created-gated daily ACE observations available")
rec_origin = ext_index.tz_convert(None).values.astype("datetime64[m]")[:, None]
rec_target = ((ext_index.tz_convert(None).values.astype("datetime64[h]")[:, None] + np.arange(1, H + 1).astype("timedelta64[h]"))
              - rec_h0) / np.timedelta64(1, "h")
rec_target = rec_target.astype(int)


def rec_observed(offset):
    selected = rec_target - offset
    in_range = (selected >= 0) & (selected < len(rec_samples))
    clipped = np.clip(selected, 0, len(rec_samples) - 1)
    valid = in_range & (rec_created[clipped] <= rec_origin) & (rec_h0 + clipped * np.timedelta64(1, "h") <= rec_origin)
    return np.where(valid, rec_samples[clipped], np.nan)


consensus = np.minimum(rec_observed(REC_LAG), rec_observed(2 * REC_LAG))
recent = np.median(np.stack([rec_observed(REC_LAG + k) for k in (0, 1, 2)]), axis=0)
earlier = np.median(np.stack([rec_observed(REC_LAG + k) for k in (6, 7, 8)]), axis=0)
EXT_VARY["rec54_consensus"] = np.where(np.isfinite(consensus), consensus, EXT_MISSING).astype(np.float32)
EXT_VARY["rec27_ramp"] = np.where(np.isfinite(recent - earlier), recent - earlier, EXT_MISSING).astype(np.float32)


def next_arrival(messages, origins):
    """origin에서 가장 가까운 살아 있는 지구 도착 예보까지 남은 시간(h)과 그 CME 속도 (7일 창, 6 h 지난 것까지)."""
    records = [r for r in parse_corpus(messages) if r["status"] == "accepted"]
    records.sort(key=lambda r: (pd.Timestamp(r["body_issue"]), r["message_id"]))
    bodies = {m["messageID"]: m.get("messageBody", "") for m in messages}
    for record in records:
        record["kinematics"] = kinematics(bodies[record["message_id"]], record["event_ids"]) if record["action"] == "forecast" else None
    out = np.full((len(origins), 2), EXT_MISSING)
    active, position = {}, 0
    for i, origin in enumerate(origins):
        while position < len(records) and pd.Timestamp(records[position]["body_issue"]) <= origin:
            record = records[position]
            ids = frozenset(record["event_ids"])
            for key, old in list(active.items()):
                if ids.intersection(old["event_ids"]):
                    del active[key]
            if record["action"] == "forecast":
                active[record["message_id"]] = record
            position += 1
        active = {k: r for k, r in active.items() if pd.Timestamp(r["body_issue"]) >= origin - pd.Timedelta(days=7)}
        pending = sorted(((pd.Timestamp(r["arrival"]) - origin) / pd.Timedelta(hours=1), r["message_id"], r)
                         for r in active.values() if pd.Timestamp(r["arrival"]) >= origin - pd.Timedelta(hours=6))
        if pending:
            hours, _, record = pending[0]
            out[i, 0] = hours
            if record["kinematics"]:
                out[i, 1] = record["kinematics"]["speed"]
    return out


arrival_state = next_arrival(donki_messages, ext_index)
# 같은 선택 규칙이므로 pub_enlil_next_arr_h, pub_cme_next_speed_kms와 같아야 함
assert np.array_equal(arrival_state.astype(np.float32), np.column_stack([EXT_CONST["pub_enlil_next_arr_h"], EXT_CONST["pub_cme_next_speed_kms"]]))
rec27 = lead_grid("rec27").astype(float)
current = one.v_lag0.to_numpy(dtype=float)[:, None]
arrival = arrival_state[:, :1]
phase = np.exp(-0.5 * ((np.arange(1, H + 1)[None, :] - arrival) / 12.) ** 2)
valid = (arrival != EXT_MISSING) & np.isfinite(rec27) & (rec27 > 0) & np.isfinite(current) & (current > 0)
EXT_VARY["cme_phase_rec27"] = np.where(valid, phase * (rec27 - current), EXT_MISSING).astype(np.float32)
print(f"ACE 일별 파일 {rec_files:,}개 | rec54_consensus 유효 {(EXT_VARY['rec54_consensus'] != EXT_MISSING).mean():.1%}")

In [ ]:
# WSA-Enlil 지구 위치 예보 속도: 공개(S3 LastModified) 후 48 h 안의 최신 실행, origin + 72 h까지 덮는 실행만


def native_hourly(seconds, speed_kms, run_utc):
    """원 표본의 완전한 UTC 1시간 칸 평균. 칸 양 끝을 감싸는 표본이 있고 간격이 max(600 s, 중앙값 × 2) 이하일 때만."""
    seconds, speeds = np.asarray(seconds, np.float64), np.asarray(speed_kms, np.float64)
    increments = np.diff(seconds)
    if seconds.ndim != 1 or len(seconds) < 2 or not np.isfinite(seconds).all() or not (increments > 0).all():
        raise ValueError("Native times must be finite and strictly increasing")
    max_gap = min(600., 2. * float(np.median(increments)))
    absolute = run_utc.timestamp() + seconds
    times, values = [], []
    for hour in range(int(np.ceil(absolute[0] / 3600)) + 1, int(np.floor(absolute[-1] / 3600)) + 1):
        start, stop = (hour - 1) * 3600, hour * 3600
        left = np.searchsorted(absolute, start, side="right")
        right = np.searchsorted(absolute, stop, side="right")
        before, after = left - 1, np.searchsorted(absolute, stop, side="left")
        value = np.nan
        if before >= 0 and after < len(absolute) and right > left:
            segment = speeds[before:after + 1]
            if np.all(np.diff(absolute[before:after + 1]) <= max_gap) and np.isfinite(segment).all() and (segment > 0).all():
                value = float(np.mean(speeds[left:right]))
        times.append(hour * 3600 * 10 ** 9)
        values.append(value)
    return np.asarray(times, np.int64), np.asarray(values, np.float64)


def enlil_run(run_id, run_utc, available, times_ns, speeds):
    if available < run_utc or len(times_ns) < 2:
        raise ValueError("bad run")
    speeds = np.where(np.isfinite(speeds) & (speeds > 0), speeds, np.nan)
    return dict(run_id=run_id, run_utc=run_utc, available=available, times=times_ns, speeds=speeds)


enlil_runs, enlil_rejected = [], 0
wsa_listing = json.loads((WSA_DIR / "listing.json").read_text())
for key, item in sorted(wsa_listing.items()):
    try:
        with netcdf_file(WSA_DIR / "files" / key, "r", mmap=False) as dataset:
            if dataset._attributes.get("label", b"").strip().lower() != b"earth":
                raise ValueError("not labelled Earth")
            if dataset.variables["TIME"].units != b"s" or dataset.variables["V1"].units != b"m/s":
                raise ValueError("units")
            run_utc = pd.Timestamp(dataset._attributes["rundate_cal"].decode(), tz="UTC")
            seconds = dataset.variables["TIME"][:].astype(np.float64)
            speed = dataset.variables["V1"][:].astype(np.float64) / 1000.
        times, values = native_hourly(seconds, speed, run_utc)
        enlil_runs.append(enlil_run(key, run_utc, pd.Timestamp(item["last_modified"]).tz_convert("UTC"), times, values))
    except (OSError, ValueError, KeyError, AttributeError):
        enlil_rejected += 1
enlil_runs.sort(key=lambda r: (r["run_utc"], r["available"], r["run_id"]), reverse=True)
enlil_speed = np.full((N_EXT, H), EXT_MISSING, np.float32)
lead_ns = np.arange(1, H + 1) * 3600 * 10 ** 9
for i, origin in enumerate(ext_index):
    o = origin.value
    for run in enlil_runs:
        if not 0 <= (origin - run["available"]) / HOUR <= 48:
            continue
        if run["times"][0] > o + lead_ns[0] or run["times"][-1] < o + lead_ns[-1]:
            continue
        positions = np.searchsorted(run["times"], o + lead_ns)
        safe = np.minimum(positions, len(run["times"]) - 1)
        ok = (positions < len(run["times"])) & (run["times"][safe] == o + lead_ns) & np.isfinite(run["speeds"][safe])
        enlil_speed[i, ok] = run["speeds"][safe[ok]].astype(np.float32)
        break
EXT_VARY["enlil_earth_speed"] = enlil_speed
print(f"WSA-Enlil 실행: S3 {len(enlil_runs)}개(거절 {enlil_rejected}) | "
      f"값이 있는 칸 {(enlil_speed != EXT_MISSING).mean():.1%}")

In [ ]:
# onset700_hazard: 700 km/s 첫 도달의 시간별 조건부 확률 -> 각 리드의 첫 도달 확률
HAZARD_CONST = ["v_lag0", "ace_speed_kms_last", "ace_density_cm3_last", "ace_density_cm3_m6", "ace_temperature_k_last",
                "ace_temperature_k_m24", "ace_bt_nt_last", "ace_bt_nt_m6", "ace_by_gsm_nt_m24", "ace_epam_p_channel3_last",
                "ace_sis_gt10_flux_last", "suvi_t035_top2_area", "suvi_t035_meridian15_equatorial_area",
                "suvi_t035_meridian30_equatorial_area", "suvi_t055_meridian15_area", "suvi_t055_meridian30_equatorial_area",
                "suvi_t055_equatorial_area", "img_age_h", "enlil_n_pending", "enlil_kp", "v_lag24", "v_lag48", "v_lag72",
                *PUB_FEATURES]
HAZARD_VARY = ["rec26", "rec27", "rec28", "rec_mean", "strip_v350_logmed", "strip_v450_logmed", "strip_v550_dark",
               "strip_v750_logmed"]
HAZARD_FEATURES = HAZARD_CONST + HAZARD_VARY
MATURITY = pd.Timedelta(hours=144)
hazard = np.full((N_EXT, H), EXT_MISSING, np.float32)
hazard_blocks = []
if not SMOKE:
    base_cols = [c for c in HAZARD_FEATURES if c not in PUB_FEATURES]
    frame = pd.read_parquet(BASE, columns=["set", "origin", "h", "target_time", "fit_F1", *base_cols])
    frame["origin"] = pd.to_datetime(frame.origin, utc=True)
    position = ext_index.get_indexer(pd.DatetimeIndex(frame.origin))
    for name in PUB_FEATURES:
        frame[name] = EXT_CONST[name][position]
    frame = frame[["set", "origin", "h", "target_time", "fit_F1", *HAZARD_FEATURES]]
    fit_origins = pd.DatetimeIndex(frame.loc[frame.fit_F1 & frame.h.eq(1), "origin"]).sort_values()
    freeze = fit_origins.max() - pd.DateOffset(months=3)
    frame = frame.sort_values(["origin", "h", "set"], ascending=[True, True, False], kind="stable")
    frame = frame.drop_duplicates(["origin", "h"], keep="first")
    assert np.array_equal(frame.h.to_numpy().reshape(-1, H), np.broadcast_to(np.arange(1, H + 1), (N_EXT, H)))
    assert pd.DatetimeIndex(frame.origin.iloc[::H]).equals(ext_index)
    current_speed = frame.v_lag0.to_numpy(dtype=np.float32).reshape(-1, H)[:, 0].copy()
    labels_before = pd.read_parquet(BASE, columns=["origin", "h", "y"],
                                    filters=[("set", "==", "train"), ("target_time", "<", freeze.to_pydatetime())])
    known = labels_before.set_index(["origin", "h"]).y
    if not known.index.is_unique:
        raise ValueError("Duplicate historical training labels")
    raw_y = known.reindex(pd.MultiIndex.from_frame(frame[["origin", "h"]])).to_numpy(dtype=np.float32).reshape(-1, H)
    valid_y = np.isfinite(raw_y) & (raw_y > 0)
    consecutive = np.logical_and.accumulate(valid_y, axis=1)
    crossing = valid_y & (raw_y >= 700)
    earlier_crossing = np.c_[np.zeros(N_EXT, bool), np.logical_or.accumulate(crossing, axis=1)[:, :-1]]
    eligible = np.isfinite(current_speed) & (current_speed > 0) & (current_speed < 700)
    risk = consecutive & ~earlier_crossing & eligible[:, None]
    labels = (crossing & risk).astype(np.float32)
    del raw_y, labels_before, known
    frame[HAZARD_FEATURES] = frame.groupby("h", sort=False)[HAZARD_FEATURES].ffill()
    matrix = frame[HAZARD_FEATURES].to_numpy(dtype=np.float32)
    matrix[matrix == EXT_MISSING] = np.nan
    matrix = np.column_stack([matrix, frame.h.to_numpy(dtype=np.float32)])
    del frame
    begin = ext_index.min().normalize()
    boundaries = sorted({begin, *pd.date_range(begin, freeze, freq="QS", tz="UTC"), freeze})
    for start, end in zip(boundaries, [*boundaries[1:], ext_index.max() + HOUR]):
        rows = np.flatnonzero((ext_index >= start) & (ext_index < end))
        if not len(rows):
            continue
        selected_risk = risk & (ext_index + MATURITY <= start)[:, None]
        cells = np.flatnonzero(selected_risk.ravel())
        target = labels.ravel()[cells]
        if not len(target) or target.min() == target.max():
            hazard_blocks.append((start, "no_two_class_history", len(target), int(target.sum()) if len(target) else 0))
            continue
        model = xgb.XGBClassifier(n_estimators=120, max_depth=3, learning_rate=.05, min_child_weight=20, reg_lambda=5,
                                  subsample=1., colsample_bytree=1., tree_method="hist", device="cpu", n_jobs=1,
                                  random_state=42, objective="binary:logistic")
        model.fit(matrix[cells], target)
        rate = model.predict_proba(matrix[(rows[:, None] * H + np.arange(H)).ravel()])[:, 1].reshape(-1, H).astype(float)
        if not np.isfinite(rate).all() or np.any((rate < 0) | (rate > 1)):
            raise ValueError("Hazards must be finite probabilities")
        probability = (np.c_[np.ones(len(rate)), np.cumprod(1.0 - rate, axis=1)[:, :-1]] * rate).astype(np.float32)
        probability[:, ~selected_risk.any(axis=0)] = EXT_MISSING
        ok = np.isfinite(current_speed[rows]) & (current_speed[rows] > 0) & (current_speed[rows] < 700)
        hazard[rows[ok]] = probability[ok]
        hazard_blocks.append((start, "fit", len(target), int(target.sum())))
        del model
    del matrix
EXT_VARY["onset700_hazard"] = hazard
print(f"onset700_hazard: 블록 {len(hazard_blocks)}개 (적합 {sum(b[1] == 'fit' for b in hazard_blocks)}) | "
      f"값이 있는 칸 {(hazard != EXT_MISSING).mean():.1%}")

In [ ]:
# 446열 표에 20열을 붙여 df.parquet로 쓰고 중간 파일을 지운다
assert set(EXT_CONST) | set(EXT_VARY) == set(EXT_ORDER) and not set(EXT_CONST) & set(EXT_VARY)
path, tmp = OUT / "df.parquet", OUT / "df.parquet.tmp"
source, writer = pq.ParquetFile(BASE), None
try:
    for i in range(source.num_row_groups):
        tbl = source.read_row_group(i)
        row = ext_index.get_indexer(pd.DatetimeIndex(tbl["origin"].to_pandas()))
        if (row < 0).any():
            raise ValueError("origin without added features")
        lead = tbl["h"].to_numpy().astype(int) - 1
        for name in EXT_ORDER:
            value = EXT_CONST[name][row] if name in EXT_CONST else EXT_VARY[name][row, lead]
            tbl = tbl.append_column(name, pa.array(np.asarray(value, np.float32), type=pa.float32()))
        if writer is None:
            writer = pq.ParquetWriter(tmp, tbl.schema, compression="zstd")
        writer.write_table(tbl)
finally:
    if writer is not None:
        writer.close()
tmp.replace(path)
BASE.unlink()
columns = tbl.schema.names
print(f"{path}: {sum(n_rows.values()):,}행 × {len(columns)}열 (추가 {len(EXT_ORDER)}열), {path.stat().st_size / 1e9:.2f} GB")

In [8]:
q_per = period_of.dropna().value_counts().sort_index()
manifest = {
    "dataset": "input-df",
    "description": "Flat (set, origin, h) model-input table built only from data-v2/raw: every raw-derived feature, "
                   "hourly train and query origins, y = ACE SWEPAM speed with status 0 (no fill), F1/F2/F3 144 h masks.",
    "built_by": "data-v2/notebooks/raw_to_input_df.ipynb",
    "built_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "smoke": SMOKE,
    "rules": {
        "time_bin": "a time t goes to hour bin floor(t, 1h) + 1h ([H-1h, H) -> H); latest record per bin; origin O uses bins <= O",
        "y": "SWEPAM bulk speed at the label time T = target_time (no shift), status == 0 and finite > 0, else NaN, no fill",
        "ace_features": "value is NaN when its status != 0 or it equals the product missing value; no fill, no log, no clip",
        "suvi": "only the frame in bin O (obs_end in [O-1h, O)); no walk-back, no stale or cover mask",
        "enlil": "runs with bin(modelCompletionTime) in [O-7d, O]",
        "added_features": "section 6b: publication-gated sources (issue/Created/Last-Modified time <= origin), -999 = missing, "
                          "ram pressure on h-wise forward-filled n and v, onset700_hazard fit on training y before 2025-09-23",
        "processing": "no availability delay, missing-value or outlier handling",
    },
    "rows": int(sum(n_rows.values())),
    "rows_by_set": n_rows,
    "columns": columns,
    "constant_features": len(CONST_COLS) + len(EXT_CONST),
    "varying_features": len(VARY_COLS) + len(EXT_VARY),
    "added_features": EXT_ORDER,
    "train_origins": len(train_origins), "train_span": [str(train_origins[0]), str(train_origins[-1])],
    "query_origins": len(query_origins), "query_span": [str(query_origins[0]), str(query_origins[-1])],
    "query_origins_by_period": {k: int(v) for k, v in q_per.items()},
    "sources": {
        "ace": {"dir": str(ACE_DIR.relative_to(DATA)), **{k: v for k, v in ace_info.items()},
                "labels": [str(ace_label.index.min()), str(ace_label.index.max())]},
        "suvi": {"dir": str(NC_DIR.relative_to(DATA)), "files_used": int(len(nc)), "files_same_bin_dropped": multi,
                 "read_errors": errors, "obs_end": [str(nc.obs_end.min()), str(nc.obs_end.max())]},
        "enlil": {"dir": str(ENLIL_DIR.relative_to(DATA)), "runs": int(len(runs))},
        "donki_notifications": {"dir": str(DONKI_DIR.relative_to(DATA)), "messages": len(donki_messages)},
        "srs": {"dir": str(SRS_DIR.relative_to(DATA)), "reports": n_srs},
        "discussion": {"dir": str(DISCUSSION_DIR.relative_to(DATA)), "reports": n_discussion},
        "ace_daily": {"dir": str(ACE_DAILY_DIR.relative_to(DATA)), "files": rec_files},
        "wsa_enlil": {"dir": str(WSA_DIR.relative_to(DATA)), "runs": len(enlil_runs), "rejected": enlil_rejected},
        "onset700_hazard": {"blocks": [[str(b[0]), b[1], b[2], b[3]] for b in hazard_blocks]},
    },
    "files": {"df.parquet": {"bytes": path.stat().st_size}},
}
(OUT / "MANIFEST.json").write_text(json.dumps(manifest, indent=1, default=str) + "\n")
print("MANIFEST.json 저장 |", manifest["rows_by_set"], "| 상수", manifest["constant_features"], "| h별", manifest["varying_features"])

MANIFEST.json 저장 | {'train': 2351880, 'query': 464904} | 상수 417 | h별 20


## 7. 검증

In [9]:
pf = pq.ParquetFile(path)
assert pf.schema_arrow.names == columns and pf.metadata.num_rows == sum(n_rows.values())
meta = pd.read_parquet(path, columns=["set", "origin", "h", "target_time", "period", "fit_F1", "fit_F2", "fit_F3", "y"])
q, tr = meta[meta["set"] == "query"], meta[meta["set"] == "train"]

# (a) origin 수와 간격: set마다 1h 연속, origin당 h = 1..72
for name, part, want in (("train", tr, train_origins), ("query", q, query_origins)):
    o = part["origin"].drop_duplicates()
    assert len(o) == len(want) and (o.diff().dropna() == HOUR).all(), name
    assert (part.groupby("origin")["h"].agg(lambda s: (s.to_numpy() == H_ALL).all())).all(), name
if not SMOKE:
    per = q.groupby("period", observed=True)["origin"].nunique().to_dict()
    assert per == {"F1": 2089, "F2": 2113, "F3": 2113}, per
print("(a) origin: train", tr["origin"].nunique(), "| query", q["origin"].nunique(), q.groupby("period", observed=True)["origin"].nunique().to_dict())

# (b) target_time = origin + h, y = target_time 라벨의 SWEPAM status 0 속도 (이동·채움 없음)
assert (meta["target_time"] - meta["origin"] == pd.to_timedelta(meta["h"], unit="h")).all()
sw = ace_raw["swepam"]
ok0 = sw["S"].eq(0) & (sw["ace_speed_kms"] > 0)
want_y = sw["ace_speed_kms"].where(ok0).reindex(pd.DatetimeIndex(meta["target_time"])).to_numpy()
assert np.array_equal(meta["y"].to_numpy(), want_y, equal_nan=True)
print(f"(b) y: target_time 라벨 status 0 속도와 일치 | NaN {meta['y'].isna().mean():.2%}")

# (c) fit 마스크: train 행만, origin + 144h <= cutoff, 학습 타깃은 그 기간 첫 query origin 이전
for p, cut in CUTOFF.items():
    used = meta[meta[f"fit_{p}"]]
    assert used["set"].eq("train").all() and (used["origin"] + pd.Timedelta(hours=MASK_HOURS) <= cut).all()
    first_q = pd.Timestamp(PERIODS[p][0])
    assert (used["target_time"] < first_q).all()
    print(f"(c) {p}: 학습 origin {used['origin'].nunique():,}개 (마지막 {used['origin'].max()}), cutoff {cut}")

# (d) 인과성: 표본 origin에서 원천 시각이 origin보다 앞서는지 raw에서 다시 확인
rng = np.random.default_rng(0)
sample = pd.DatetimeIndex(rng.choice(all_origins, size=min(300, len(all_origins)), replace=False))
cols_d = ["origin", "h", "ace_speed_kms_last", "ace_bt_nt_m6", "img_age_h", "enlil_n_pending", "rec27", "strip_v450_logmed"]
part = (pd.read_parquet(path, columns=cols_d, filters=[("h", "=", 1)]).drop_duplicates("origin")
        .set_index("origin").loc[sample].sort_index())
lab = ace_label.reindex(part.index - HOUR)                          # 칸 O = 라벨 O-1h
assert np.allclose(part["ace_speed_kms_last"], lab["ace_speed_kms"], equal_nan=True)
win = [ace_label["ace_bt_nt"].reindex(pd.date_range(o - 6 * HOUR, o - HOUR, freq="h")).mean() for o in part.index]
assert np.allclose(part["ace_bt_nt_m6"], np.asarray(win, dtype=np.float32), equal_nan=True)
age = part["img_age_h"].dropna()
assert ((age > 0) & (age <= 1)).all()                               # obs_end in [O-1h, O)
n_pend = [int(((runs["bin"] <= o) & (runs["bin"] >= o - pd.Timedelta(days=7)) &
               ((runs["arrival"] - o) / HOUR >= -6)).sum()) for o in part.index]
assert np.array_equal(part["enlil_n_pending"].to_numpy(), np.asarray(n_pend, dtype=np.float32))
rec_src = ace_label["ace_speed_kms"].reindex(part.index + HOUR - pd.Timedelta(days=27) - HOUR)
assert np.allclose(part["rec27"], rec_src, equal_nan=True) and (part.index + HOUR - pd.Timedelta(days=27) <= part.index).all()
print(f"(d) 표본 {len(part)} origin: ACE last·m6, img_age_h ∈ (0, 1], enlil_n_pending, rec27이 raw 재계산과 일치")

# (e) 이름: ch 토큰 없음, 열 중복 없음
assert not [c for c in columns if re.search(r"(^|_)ch(_|$)", c.lower())]
assert len(set(columns)) == len(columns)
print("(e) 열", len(columns), "| ch 이름 없음")

# (f) 결측 비율 (처리하지 않은 그대로)
nan_rate = pd.read_parquet(path, columns=CONST_COLS + VARY_COLS, filters=[("h", "=", 1)]).isna().mean()
display(nan_rate.groupby(lambda c: "ace" if c.startswith(("ace_", "v_", "speed_", "epam_")) else
                         "suvi" if c.startswith(("suvi_", "img_", "strip_")) else "rec" if c.startswith("rec") else "enlil")
        .describe().round(3))

# (g) 추가 20열: 순서, float32, 유한값 또는 -999 (pub_enlil_glancing만 -1/0/1)
schema = pq.ParquetFile(path).schema_arrow
assert columns[-len(EXT_ORDER):] == EXT_ORDER and all(schema.field(n).type == pa.float32() for n in EXT_ORDER)
ext_check = pd.read_parquet(path, columns=EXT_ORDER)
assert np.isfinite(ext_check.to_numpy()).all(), "NaN/inf in an added column"
print("(g) 추가 열", len(EXT_ORDER), "| -999 비율", ext_check.eq(EXT_MISSING).mean().round(3).to_dict())


(a) origin: train 32665 | query 6457 {'F1': 2089, 'F2': 2113, 'F3': 2113}


(b) y: target_time 라벨 status 0 속도와 일치 | NaN 32.18%
(c) F1: 학습 origin 28,297개 (마지막 2025-12-23 00:00:00+00:00), cutoff 2025-12-29 00:00:00+00:00


(c) F2: 학습 origin 30,457개 (마지막 2026-03-23 00:00:00+00:00), cutoff 2026-03-29 00:00:00+00:00
(c) F3: 학습 origin 32,665개 (마지막 2026-06-23 00:00:00+00:00), cutoff 2026-06-29 00:00:00+00:00


(d) 표본 300 origin: ACE last·m6, img_age_h ∈ (0, 1], enlil_n_pending, rec27이 raw 재계산과 일치
(e) 열 446 | ch 이름 없음


,count,mean,std,min,25%,50%,75%,max
ace,83.0,0.114,0.208,0.000,0.000,0.009,0.168,1.000
enlil,6.0,0.272,0.213,0.000,0.088,0.388,0.427,0.427
rec,4.0,0.279,0.087,0.148,0.279,0.323,0.323,0.323
suvi,344.0,0.066,0.067,0.053,0.053,0.053,0.053,0.466


## 8. 사용법

```python
import pandas as pd
DF = "/home/t-lab01/sunrun/data-v2/input/df.parquet"
cols = ["set", "origin", "h", "period", "fit_F1", "y", "v_lag0", "suvi_t035_equatorial_area", "strip_v450_logmed"]
df = pd.read_parquet(DF, columns=cols)                      # 전체 약 2.8M행 x 466열이므로 필요한 열만 읽기
fit_f1 = df[df.fit_F1]                                       # F1 학습 행
query_f1 = df[(df.set == "query") & (df.period == "F1")]    # F1 예측 origin 2,089개 x 72h
```

## 9. 한계와 주의

- 값이 예전 df(2026-11-01)와 다릅니다.
  - 시간 칸 규칙 때문에 ACE 칸 O는 라벨 O−1h 행입니다.
  - SUVI·ENLIL에 되돌아가기와 마스크가 없습니다.
  - `y`는 status 0만 쓰고 채우지 않습니다.
  - train origin이 3h가 아니라 1h 간격입니다.
  - EPAM/SIS에 log10을 쓰지 않습니다.
  - 예전 `ch_ch_*` 열은 `suvi_*`로 바뀌었고 `img_stale`은 없습니다.
- `research/e2000fs_calib_point4_144h/experiment.py`는 예전 열 이름(`ch_ch_*`)을 읽으므로 그대로는 실행되지 않습니다.
- 원문은 받은 시점의 최신 값입니다. 각 시점에 실제로 공개돼 있던 값인지는 검증되지 않았습니다.
- 2022-08-22~11-21에는 SUVI 파일이 없습니다. 그 칸의 SUVI 피처는 NaN입니다.
- 6b절 원천의 `LastModified`·`Last-Modified`는 처음 받은 값을 유지합니다. 원천이 다시 올린 뒤 새로 받으면 값이 달라질 수 있습니다.
- 생성 파일 `data-v2/input/{df.parquet, MANIFEST.json}`은 데이터라서 git에 올리지 않습니다.